# Test 8 — Replication + extension of Jang et al. (2025) on full BEIR

**Paper.** Jang, Y., Hong, S., Son, J., Park, S., Park, C., Lim, H. (2025). *From Ambiguity to
Accuracy: The Transformative Effect of Coreference Resolution on Retrieval-Augmented Generation
systems.* [arXiv:2507.07847](https://arxiv.org/abs/2507.07847) v3. Korea University + NAVER.

**Their two claims.** (a) Coreference resolution consistently improves dense retrieval.
(b) **Mean pooling gains more than CLS or last-token pooling**, because mean pooling weights all
tokens equally, so replacing a pronoun with a name adds real semantic content. Their effects are
tiny — overall nDCG moves +0.001 to +0.006, `bge-large-en-v1.5` (CLS) shows 0.789 → 0.789, several
cells decrease — and **they report no confidence intervals**.

**What this notebook adds.**

| # | Contribution | Why it is missing today |
|---|---|---|
| 1 | **Pooling as a variable** | Tests 1–7 in this repo are CLS-only, so the paper's mechanism was never exercised |
| 2 | **Model generation as a variable** — *underpowered here* | The *mean-pooling ladder* holds pooling constant so generation is isolated. `embeddinggemma-300m` (the 2025 rung) was dropped for GPU cost, so the ladder spans 2021 → 2022 only and this contribution is **registered but not answered in this run** |
| 3 | **Stratification** | The paper averages over all queries; coref can only help a query whose gold passage hides an entity the query names |
| 4 | **Bootstrap confidence intervals** | Neither the paper nor this repo has any, anywhere |
| 5 | **Controlled LingMess vs LLM coref** | The repo README lists this as an open gap: never run on the same data |

**Three variables out of one coref pass.** The LLM coref stage is the expensive one and it runs
once, over the whole corpus. Embedding passes cost minutes, so every extra encoder is nearly free.
That buys three factors from a single expensive stage: **coref method** (none / LingMess / LLM),
**pooling strategy** (mean / CLS / last-token), and **model generation** (2021 → 2022 only in this
run, and therefore not answerable — see the roster note below).

---

## Everything runs here. Nothing is done off-notebook.

Datasets are pulled from HuggingFace, coref is computed here, retrieval and statistics are computed
here, and every artifact is written to `/kaggle/working/test-8-data/`.

```
STAGE -1 install, in-process           -> no kernel restart, ever
STAGE 0  validate the LLM coref path   -> stage0_validation.json      [GATE: thresholds]
STAGE 1  load BEIR + stratify          -> stage1_*__{dataset}.json
STAGE 2  LingMess coref, whole corpus  -> stage2_lingmess__{ds}.json  [stratum: recorded]
STAGE 3  LLM coref, whole corpus       -> stage3_llm__{ds}.json       [ckpt every 200]
STAGE 4  embed x4, score, write RAW    -> per_query_results.jsonl     [floor: quarantine]
ANALYSIS regress / test / plot         -> test-8-results.md + PNGs    [CPU ONLY]
SUMMARY  what ran, what was skipped    -> RESULTS.md + run_summary.json
```

**Built to be pushed and left alone.** There is no manual kernel restart and no cell that waits for
a human. Under a Kaggle batch job the Stage 0 gate is decided by declared numeric thresholds
(printed and recorded), a thin stratum is recorded as a finding instead of halting, and a model
below its sanity floor is quarantined — its rows are flagged and excluded from every table — rather
than killing the models that had not run yet. A session budget keeps the run inside Kaggle's 12h
ceiling by flushing and stopping cleanly, so a second push resumes from the checkpoints instead of
recomputing.

**The run and the analysis are separate.** Stage 4 computes no findings: it writes one raw row per
(model, dataset, condition, retrieval, query) — including that query's top-50 doc ids — and appends
them **after every model**, `flush()` + `fsync()`, before recording the model as done. A session
that dies at model 4 still leaves models 1–3 fully analysable. The final cell reads that one file,
needs no GPU, and is where every table and plot in this notebook comes from.

**Every stage checkpoints.** Each one checks whether its output already exists: if yes it loads and
skips, if no it computes and saves — atomically, so a timeout mid-write cannot leave a corrupt file
that later loads as garbage. Kaggle allows no concurrent GPU sessions, so a crash or a 12h timeout
resumes instead of starting over.

**Never two models on the GPU at once.** Free VRAM is asserted and logged before every load, and
`del model; gc.collect(); torch.cuda.empty_cache()` runs after every one.

---

## Benchmark: full BEIR `scifact` + `nfcorpus`

| | scifact | nfcorpus |
|---|---|---|
| corpus | 5,183 | 3,633 |
| test queries | ~300 | ~323 |
| gold per query | ~1.1 | ~38 |
| relevance | binary | **graded 0/1/2** |
| role here | carries the **stratified** arm | carries the **aggregate** arm |

These are **full standard BEIR sets, not NanoBEIR**. NanoBEIR is a 50-query screening subset; no
paper reports on it, and 50 queries cannot support a confidence interval worth printing. These two
are also the only full-BEIR sets small enough to LLM-coref end to end on a free T4 (the next step
up, `scidocs`, is 25,657 passages; `nq` is 2.68M). Together they give **8,816 passages and ~623
queries**. **Results are reported per dataset and never pooled**: the two sets differ in gold
density (~1.1 vs ~38 gold passages per query) and in relevance grading, so a pooled average would
average two different measurement problems.

### Three data hazards this notebook handles explicitly

1. **`scifact` qrels use `int64` ids while its corpus uses strings.** A naive join silently matches
   nothing. Every id is normalised with `str()` on both sides, and Stage 1 asserts that every gold
   id resolves.
2. **`nfcorpus` relevance is graded 0/1/2, `scifact` is binary.** Flattening grades to 1 would
   quietly mis-measure nDCG. Grades are preserved and passed to `pytrec_eval` as-is; "gold" means
   `score > 0`.
3. **Both corpora are `title` + `text`.** BEIR convention concatenates them, and that concatenated
   string is what gets coref-resolved and embedded. Stage 1 logs how often the title carries the
   entity that the body then pronominalises.

### A prediction registered before the run

The paper's only true IR benchmark is NanoSCIDOCS, and on it **both their encoder models go
flat-or-down** at @3/@5 (`bge-large` 0.364 → 0.359; `e5-large-v2` 0.359 → 0.352). Their positive
averages come from BELEBELE / SQuAD2.0 / BoolQ, which are QA sets adapted into retrieval. This
notebook runs four **encoders** on scientific IR data, so **the paper's own data predicts a null or
a small negative here.** Written down now so the result cannot be reinterpreted afterwards.

**And the honest risk:** scientific abstracts use far fewer pronouns than Wikipedia. If pronoun
density is low, coref has little to change and any null is *by construction*, not a finding.
Stage 1 measures and prints pronoun density per dataset **before** the expensive stage, so the run
can be abandoned early.

### A second hypothesis, registered before the run — and tested *against*, not assumed

> **Hypothesis.** Newer embedding models capture enough context on their own that coreference
> resolution yields **smaller** gains than it does on older models.

**Jang et al.'s own Table 1 mildly contradicts this.** Their largest and newest model (LLM2Vec, 8B)
showed their **biggest** gain (+0.006), while the older `bge-large` encoder showed 0.789 → 0.789 —
no change at all. So the hypothesis is registered as something this notebook **tests against**,
not something it assumes or expects to confirm. Either direction is a publishable outcome, and both
are written down here before any number exists.

The design that makes it answerable is the **mean-pooling ladder**: three models that all use mean
pooling, so *pooling is held constant and only generation moves*.

| role | model | year | params | pooling |
|---|---|---|---|---|
| ladder | `sentence-transformers/all-MiniLM-L6-v2` | 2021 | 22M | mean |
| ladder | `intfloat/e5-small-v2` | 2022 | 33M | mean |
| ~~ladder~~ | ~~`google/embeddinggemma-300m`~~ | ~~2025~~ | ~~300M~~ | **dropped for GPU cost** |
| pooling contrast | `BAAI/bge-small-en-v1.5` | 2023 | 33M | **CLS** |
| pooling contrast | `Qwen/Qwen3-Embedding-0.6B` | 2025 | 596M | **last-token** |

**What dropping the 2025 rung costs.** The paper's own claim is the pooling one, and it survives
intact: mean (MiniLM, e5) vs CLS (bge) vs last-token (Qwen3) are all still measured. The *ladder*
does not survive: two tiny models three years apart cannot say whether newer encoders gain less
from coreference, so that hypothesis is **registered but not answered here**, and the analysis
prints that rather than reporting a two-rung ladder as if it had settled the question.

Older encoders tend to mean-pool and newer ones vary, so comparing MiniLM against `bge` would move
generation **and** pooling at the same time and the resulting number would be uninterpretable. The
ladder is the only part of this roster that can speak to the hypothesis; the other two models speak
to the paper's pooling claim.

## Stage −1 · Install — in-process, no kernel restart

`fastcoref`'s LingMess targets the transformers 4.x API; Kaggle ships 5.x, whose
`all_tied_weights_keys` property breaks LingMess loading. So transformers is pinned `<5`, exactly as
in tests 2–3 of this repo.

**This cell installs and then takes the new versions inside the same process.** It runs before
anything imports transformers, purges the module cache, and asserts the pin actually took. There is
no manual restart anywhere in this notebook: it is designed to be pushed as a Kaggle batch job and
left alone.


In [ ]:
import os, sys, subprocess, importlib, site

MARKER = "/kaggle/working/.test8_install_done"

def _pkgs_ok():
    """After a resumed session the marker persists; confirm the packages really are importable."""
    try:
        import transformers, fastcoref, pytrec_eval, bitsandbytes, spacy  # noqa: F401
        import datasets, rank_bm25                                        # noqa: F401
        import scipy, matplotlib                                          # noqa: F401
        return int(transformers.__version__.split(".")[0]) < 5
    except Exception:
        return False

if os.path.exists(MARKER) and _pkgs_ok():
    import transformers
    print(f"install: SKIP (marker present, transformers {transformers.__version__})")
else:
    if os.path.exists(MARKER):
        os.remove(MARKER)
        print("install: marker was stale (fresh session) - reinstalling")
    for c in [
        [sys.executable, "-m", "pip", "install", "-q",
         "datasets", "pytrec_eval", "tqdm", "pandas", "numpy", "rank_bm25",
         # scipy/matplotlib ship with the Kaggle image; listed so the analysis cell cannot
         # fail on a stripped environment. No-ops when they are already present.
         "scipy", "matplotlib"],
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers<5", "fastcoref", "accelerate", "bitsandbytes"],
        [sys.executable, "-m", "spacy", "download", "en_core_web_sm"],
    ]:
        print("$", " ".join(c[2:]))
        subprocess.run(c, check=True)
    open(MARKER, "w").write("ok")

    # ---------------- take the new versions WITHOUT restarting the kernel ----------------
    # A restart is only needed for a module that is ALREADY imported: Python caches it in
    # sys.modules, and a later `import` hands back the stale object no matter what is now on
    # disk. This cell is the first thing that runs, so at most the probe above imported
    # anything. Dropping those entries and invalidating the finder cache makes the next
    # import read the freshly installed files. spacy prints its own "Restart to reload
    # dependencies" notice during the model download - that notice is written for the
    # general case and is handled here by the same purge.
    #
    # The assert at the bottom is what proves it worked. If the pin somehow did not take,
    # this halts HERE with an explanation, rather than surfacing hours later inside LingMess.
    _stale = [m for m in list(sys.modules)
              if m.split(".")[0] in ("transformers", "tokenizers", "fastcoref", "accelerate",
                                     "bitsandbytes", "datasets", "huggingface_hub", "spacy",
                                     "thinc", "peft", "en_core_web_sm")]
    for _m in _stale:
        del sys.modules[_m]
    importlib.invalidate_caches()
    try:                                   # pip targets the user site on some images
        _usp = site.getusersitepackages()
        if isinstance(_usp, str) and os.path.isdir(_usp) and _usp not in sys.path:
            sys.path.insert(0, _usp)
    except Exception:
        pass
    print(f"install: dropped {len(_stale)} cached module(s), re-importing in-process")

    import transformers
    print(f"install: DONE in-process | transformers {transformers.__version__}")
    if int(transformers.__version__.split(".")[0]) >= 5:
        raise RuntimeError(
            "\n\n=========================================================================\n"
            f"  transformers is still {transformers.__version__} after the pin to <5.\n"
            "  LingMess cannot load on 5.x. This is the ONE case that needs a manual\n"
            "  kernel restart: restart once, then Run All - every finished stage loads\n"
            "  from its checkpoint.\n"
            "=========================================================================")


## Stage −1b · Config, seeds, gates

**This notebook is built to run unattended.** Under a Kaggle batch job (`Save & Run All`) it detects
`KAGGLE_KERNEL_RUN_TYPE=Batch` and runs end to end with no human in the loop. The two gates below
still exist — they are the reason a bad LLM coref path cannot quietly consume a GPU session — but in
batch mode they are decided by **declared numeric thresholds** instead of by a person flipping a
bool, and every decision is printed and written into `stage0_validation.json` and `run_summary.json`.

* `STAGE0_APPROVED` — Stage 0 validates that the LLM does coreference resolution rather than
  paraphrase or invention. Interactively it waits for you. In batch it is auto-approved **only** if
  the LLM actually changed passages, its generation-failure rate is under `STAGE0_MAX_FAIL_RATE`,
  and its hallucinated-antecedent rate is under `STAGE0_MAX_HALLUC_RATE`. Outside those bounds the
  run halts, exactly as before.
* `STRATUM_GATE_OVERRIDE` — a thin confirmed stratum is **recorded as a finding**, not treated as
  an error. The aggregate arm is the paper's own condition and does not depend on that stratum, so
  the run continues and the analysis carries the stratum as a diagnostic.

A third control, `SESSION_BUDGET_H`, keeps the run inside Kaggle's 12h ceiling: expensive stages
check it, flush their checkpoints, and stop cleanly with a resume banner rather than being killed
mid-write.


In [ ]:
import os, sys, gc, json, time, re, math, random, hashlib, logging, tempfile
import numpy as np

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("coref_v8")

# ------------------------------------------------------------------ run mode
# BATCH is what makes this notebook pushable: Kaggle sets KAGGLE_KERNEL_RUN_TYPE=Batch for a
# "Save & Run All" job. In batch there is nobody to answer a prompt, so the gates below decide
# themselves from declared thresholds, and progress bars are suppressed - a tqdm bar writes a
# carriage-return line per update, which in a batch log is megabytes of noise that can get the
# log truncated. Set TEST8_BATCH=1 or 0 to force either mode.
BATCH = os.environ.get("TEST8_BATCH", "").strip() or (
    "1" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "") == "Batch" else "0")
BATCH = BATCH == "1"
if BATCH:
    os.environ["TQDM_DISABLE"] = "1"          # honoured by tqdm >= 4.66 via envwrap
os.environ.setdefault("TQDM_MININTERVAL", "30")

# ------------------------------------------------------------------ time budget
# Kaggle kills a session at 12h. Expensive stages check the remaining time, flush, and stop
# cleanly instead of being killed mid-write. The margin leaves room to write checkpoints.
RUN_STARTED_AT    = time.time()
SESSION_BUDGET_H  = float(os.environ.get("TEST8_BUDGET_H", "11.2"))
RUN_DEADLINE      = RUN_STARTED_AT + SESSION_BUDGET_H * 3600
DEADLINE_MARGIN_S = 600.0                     # stop taking new work with 10 min left
for n in ("fastcoref", "transformers", "datasets", "urllib3", "filelock", "httpx"):
    logging.getLogger(n).setLevel(logging.WARNING)
try:
    import datasets as _ds; _ds.disable_progress_bars()
except Exception:
    pass

# ------------------------------------------------------------------ seeds
SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# ------------------------------------------------------------------ paths
WORK = "/kaggle/working"
DATA = os.path.join(WORK, "test-8-data")
os.makedirs(DATA, exist_ok=True)

# The ONE artifact the analysis reads. Stage 4 appends to it after every model and fsyncs, so
# a session that dies at model 4 still leaves models 1-3 fully analysable.
PER_QUERY_JSONL = os.path.join(DATA, "per_query_results.jsonl")
DONE_MODELS_CK  = "stage4_done_models.json"

# ------------------------------------------------------------------ GATES
# Interactive: STAGE0_APPROVED stays False and the notebook stops after the Stage 0 report so a
# human can read it. Batch: there is nobody to read it, so the same decision is made from the
# thresholds below and recorded. The gate is not removed - it is made explicit and machine-checked.
STAGE0_APPROVED       = None if BATCH else False   # None = decide from the thresholds below
STRATUM_GATE_OVERRIDE = True    # a thin stratum is a FINDING, not an error - it never halts

STAGE0_MAX_HALLUC_RATE = 0.35   # hallucinated antecedents, as a share of CHANGED passages
STAGE0_MAX_FAIL_RATE   = 0.20   # generation failures, as a share of all passages
STAGE0_MIN_CHANGED     = 1      # the LLM must actually rewrite something, or the path is dead

# ------------------------------------------------------------------ Stage 3 anaphor skip
# The Table 5 prompt resolves pronouns and anaphora. A passage containing no pronoun at all has
# nothing for it to resolve, so sending it to the LLM buys an identity transform at full GPU
# cost. Skipping those is the largest saving available in Stage 3 - but it is an ASSUMPTION, so
# it is not taken on faith: Stage 0 measures how many pronoun-free test-6 chunks this model
# changed anyway, and the skip activates only if that count is zero. Otherwise the measured
# miss rate is printed, recorded in stage0_validation.json, and the skip stays OFF.
SKIP_NO_ANAPHOR = True          # request the optimisation; Stage 0 decides if it activates

# ------------------------------------------------------------------ resume guard
# SET THIS True ON A RESUME PUSH. A Kaggle batch job starts with an empty /kaggle/working, so
# a resume only works if the previous run's output is attached as an INPUT. Inputs are a
# per-notebook setting: they do NOT survive importing the notebook, and a fresh import starts
# with none. When that is missed, nothing errors - the run just silently recomputes every
# stage, and the cost is only discovered hours later. With this True the run halts in the
# first seconds instead. Leave False for a genuine cold start.
EXPECT_SEED_CHECKPOINTS = True

# ------------------------------------------------------------------ benchmark
# FULL BEIR, not NanoBEIR. NanoBEIR is a 50-query screening subset that no paper reports on;
# these are the standard published sets, and the only two small enough to LLM-coref end to
# end on a free T4. Results are reported PER DATASET. Nothing is pooled across them.
DATASETS = {
    "scifact":  {"corpus_repo": "BeIR/scifact",  "qrels_repo": "BeIR/scifact-qrels",
                 "split": "test", "expect_corpus": 5183},
    "nfcorpus": {"corpus_repo": "BeIR/nfcorpus", "qrels_repo": "BeIR/nfcorpus-qrels",
                 "split": "test", "expect_corpus": 3633},
}
DS_NAMES = list(DATASETS)

# ------------------------------------------------------------------ models
# DELIBERATE DEPARTURE from the paper. Jang et al. Appendix B use Qwen2.5-7B-Instruct; this
# notebook uses the 3B instruct model of the SAME family, with the SAME Table 5 prompt and the
# SAME chat template, in fp16 rather than 4-bit NF4. Two reasons, both about fitting a free-tier
# session: 3B instead of 7B, and fp16 instead of NF4 - bitsandbytes dequantises every weight on
# every forward pass, which on an sm75 T4 makes 4-bit inference SLOWER than fp16 for a model
# that already fits. 3B fp16 is ~6.2 GB, leaving KV-cache headroom on a 15 GB T4.
#
# The whole corpus is rewritten by this one model, so no comparison inside this notebook is
# confounded - but this is NO LONGER A STRICT REPLICATION of their Appendix B configuration,
# and it is labelled as such in every stage output. STAGE 0 is what makes the substitution
# defensible rather than merely convenient: it measures THIS model's hallucination rate and
# exact-match against test-6's manual gold BEFORE Stage 3 spends anything, and halts the run
# if the coref path does not hold up.
LLM_MODEL       = "Qwen/Qwen2.5-3B-Instruct"
LLM_PAPER_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # what the paper used; recorded for honesty
LLM_LOAD_4BIT   = False                       # fp16; set True to restore NF4 quantisation
LLM_VRAM_NEED   = 7.0                         # GB free required before a load (fp16 3B + KV)
# Decoding at small batch is memory-bandwidth bound, so throughput scales almost linearly
# with batch until VRAM runs out. Stage 3 sizes each batch from a shared token budget and
# steps that budget down on OOM, so an over-estimate costs a retry, not the run.
LLM_BATCH      = 32
LLM_CKPT_EVERY = 200
LLM_MAX_NEW    = 1024

EMBED_BATCH, MAX_LEN = 128, 512   # default batch; the large encoder overrides it below

# ---- four encoders, CHEAPEST FIRST -------------------------------------------------------
# Run order is deliberate: MiniLM -> bge -> e5 -> Qwen3. The three small encoders finish in
# minutes, so a misconfiguration trips the sanity floor early, and their per-query rows are
# already on disk before the expensive one starts.
#
# DROPPED FOR COST: google/embeddinggemma-300m (2025, 300M, mean pooling, fp32) was the 2025
# rung of the mean-pooling ladder and roughly a third of Stage 4's GPU time. Removing it keeps
# THE PAPER'S OWN CLAIM fully testable - mean (MiniLM, e5) vs CLS (bge) vs last-token (Qwen3)
# are all still present - but it guts this repo's SECOND, self-registered hypothesis: the
# ladder now spans only 2021 -> 2022, both tiny models, which cannot speak to whether newer
# encoders gain less from coreference. That hypothesis is therefore REGISTERED BUT NOT
# ANSWERED here, and the analysis says so rather than reporting a two-rung ladder as if it
# had settled anything.
#
# MEAN-POOLING LADDER (pooling held CONSTANT, generation varies) - UNDERPOWERED, 2 rungs:
#     all-MiniLM-L6-v2 (2021, 22M) -> e5-small-v2 (2022, 33M)
# POOLING CONTRAST (the paper's mechanism) - COMPLETE:
#     mean (MiniLM, e5)  vs  bge-small-en-v1.5 (CLS)  vs  Qwen3-Embedding-0.6B (last-token)
#
# `prefix_scheme` names the input convention each model was TRAINED with. Getting one wrong
# does not raise - it silently costs several points of recall - so every scheme is named,
# applied explicitly, logged per model at load time, and written into every row it produced.
#
# `dtype`: fp16 on the T4 for every encoder in this roster. A model whose activations can
# overflow fp16 to NaN must declare fp32 here instead - embeddinggemma was the one such model,
# and it is the reason this field exists at all.
#
# `cost_weight`: rough relative encode cost, used ONLY for the wall-clock projection printed
# after model 1. It is a declared estimate, not a measurement.
QWEN3_TASK = "Given a web search query, retrieve relevant passages that answer the query"

EMBED_MODELS = {
    "all-MiniLM-L6-v2": {
        "hf": "sentence-transformers/all-MiniLM-L6-v2", "pooling": "mean", "dtype": "fp16",
        "role": "ladder", "year": 2021, "params_m": 22, "cost_weight": 1.0,
        "prefix_scheme": "none",              # symmetric ST model: no prefixes, by design
        "query_prefix": "", "passage_prefix": "",
        "append_eos": False, "batch": EMBED_BATCH, "vram_need": 1.0, "floor_hint": ""},
    "bge-small-en-v1.5": {
        "hf": "BAAI/bge-small-en-v1.5", "pooling": "cls", "dtype": "fp16",
        "role": "pooling-contrast", "year": 2023, "params_m": 33, "cost_weight": 1.5,
        # DELIBERATE DEPARTURE from "bge needs no prefix": the bge-small-en-v1.5 model card
        # does specify a query-side instruction for short-query retrieval (passages take
        # none). It is applied here because it is this model's documented convention and it
        # is what tests 1-7 of this repo used, so the numbers stay comparable. Flip
        # BGE_QUERY_INSTRUCTION below to run the bare no-prefix variant instead.
        "prefix_scheme": "bge_query_instruction",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
        "passage_prefix": "",
        "append_eos": False, "batch": EMBED_BATCH, "vram_need": 1.0, "floor_hint": ""},
    "e5-small-v2": {
        "hf": "intfloat/e5-small-v2", "pooling": "mean", "dtype": "fp16",
        "role": "ladder", "year": 2022, "params_m": 33, "cost_weight": 1.5,
        "prefix_scheme": "e5_query_passage",  # REQUIRED by e5-*; omitting it degrades badly
        "query_prefix": "query: ", "passage_prefix": "passage: ",
        "append_eos": False, "batch": EMBED_BATCH, "vram_need": 1.0,
        "floor_hint": "e5 collapses without its literal query:/passage: prefixes."},
    "Qwen3-Embedding-0.6B": {
        "hf": "Qwen/Qwen3-Embedding-0.6B", "pooling": "last", "dtype": "fp16",
        "role": "pooling-contrast", "year": 2025, "params_m": 596, "cost_weight": 22.0,
        # REQUIRED: queries are wrapped as Instruct: <task> / Query: <query>; documents get
        # nothing. Omitting the instruction measurably drops performance. Last-token pooling
        # additionally needs an EOS at the end of every sequence, and right-side padding.
        "prefix_scheme": "qwen3_instruct_query",
        "query_prefix": "Instruct: " + QWEN3_TASK + "\nQuery: ", "passage_prefix": "",
        "append_eos": True, "batch": 16, "vram_need": 4.0,
        "floor_hint": ("check the Instruct/Query wrapper, the appended EOS, and that padding "
                       "is right-side so last-token pooling indexes a real token.")},
}

BGE_QUERY_INSTRUCTION = True
if not BGE_QUERY_INSTRUCTION:
    EMBED_MODELS["bge-small-en-v1.5"].update(prefix_scheme="none", query_prefix="")

# condition = baseline | coref. `original` is the baseline; the two coref methods are the
# controlled A/B, and each coref row is paired to the baseline row for the same query.
VARIANTS       = ["original", "lingmess", "llm"]
COREF_VARIANTS = ["lingmess", "llm"]

# ------------------------------------------------------------------ retrieval / stats
TOP_K, RRF_K, DENSE_WEIGHT, RUN_DEPTH = 5, 60, 0.7, 100
SAVE_TOPN    = 50        # top-50 doc ids saved per query row, so any metric is recomputable
MEASURES     = {"ndcg_cut_10", "recall_10", "recip_rank"}
METRIC_NAMES = ["ndcg_cut_10", "recall_10", "recip_rank"]
PRIMARY_METRIC = "ndcg_cut_10"
BOOTSTRAP_N  = 10000     # slope / rho / mean-delta intervals, seed 42
MIN_CRITICAL = 10        # confirmed-critical floor for the stratified arm

# HALTING SANITY FLOOR, per dataset. A model below its floor on the ORIGINAL (non-coref)
# corpus is misconfigured - almost always a wrong or missing prompt prefix - and every coref
# delta measured on top of it would be noise around a broken baseline. nfcorpus is graded and
# carries ~38 gold passages per query, which caps Recall@10 near 10/38, so it is floored on
# nDCG@10 instead.
SANITY_FLOOR = {"scifact":  ("recall_10",   0.50),
                "nfcorpus": ("ndcg_cut_10", 0.25)}

# ------------------------------------------------------------------ Stage 0 reference
# test-6 of this repo, pinned to an immutable commit with SHA-256 asserts.
T6_COMMIT = "99e9918b3bcdfda6182802ddcb2cfa7758dee8de"
T6_RAW    = f"https://raw.githubusercontent.com/altamash203/coref-rag-eval/{T6_COMMIT}/test-6/test-6-data"
# SHA-256 of the bytes raw.githubusercontent SERVES, i.e. the LF blobs as committed.
# Do NOT regenerate these from a Windows working copy: core.autocrlf checks these files out
# as CRLF, which hashes differently and fails Stage 0 with a mismatch that looks like
# tampering but is only line endings.
T6_SHA = {
    "original_chunks.json": "d151d6bd9096715e4ac633622d6dfd5248359d77edd8fbe34f354294b433e1cd",
    "coref_chunks.json":    "3aefa78ed2cb417432b0a65005963091b45aba755de521b31311438deb6b240a",
}

log.info("Config | datasets=%s (reported per dataset, never pooled) | seed=%d | bootstrap=%d",
         DS_NAMES, SEED, BOOTSTRAP_N)
log.info("Config | llm=%s %s greedy (the paper used %s) | anaphor-skip requested=%s",
         LLM_MODEL, "4-bit nf4" if LLM_LOAD_4BIT else "fp16", LLM_PAPER_MODEL, SKIP_NO_ANAPHOR)
log.info("Config | run mode=%s | session budget %.1f h",
         "BATCH" if BATCH else "INTERACTIVE", SESSION_BUDGET_H)
for _m, _c in EMBED_MODELS.items():
    log.info("Config | embed %-22s %-4s pooling | %-5s | prefix_scheme=%-22s | %dM (%d)",
             _m, _c["pooling"], _c["dtype"], _c["prefix_scheme"], _c["params_m"], _c["year"])
log.info("Config | run order (cheapest first) = %s", list(EMBED_MODELS))
log.info("Config | sanity floor per dataset: %s (quarantining, not halting)",
         SANITY_FLOOR)
log.info("Config | per-query rows -> %s", PER_QUERY_JSONL)
log.info("GATES  | STAGE0_APPROVED=%s (None = decide from thresholds) | halluc<=%.2f "
         "fail<=%.2f | stratum gate never halts",
         STAGE0_APPROVED, STAGE0_MAX_HALLUC_RATE, STAGE0_MAX_FAIL_RATE)

### Verification and checkpoint plumbing

`check(...)` **raises and halts**. Nothing warns and continues. Every call logs a `CHECK PASS` /
`CHECK FAIL` line, so the log alone shows what was verified, and the full list is saved into
`stage4_results.json`.

In [ ]:
class CheckFailed(AssertionError): pass
_CHECKS = []

def check(cond, label, detail=""):
    ok = bool(cond)
    _CHECKS.append({"label": label, "pass": ok, "detail": str(detail)[:400]})
    log.info("CHECK %-4s | %s%s", "PASS" if ok else "FAIL", label, f" | {detail}" if detail else "")
    if not ok:
        raise CheckFailed(f"{label} | {detail}")
    return True

def checks_summary():
    bad = [c for c in _CHECKS if not c["pass"]]
    log.info("CHECKS: %d run, %d failed", len(_CHECKS), len(bad))
    return {"n": len(_CHECKS), "failed": len(bad), "checks": _CHECKS}

# ------------------------------------------------------------------ checkpoints
def _p(name): return os.path.join(DATA, name)

# A Kaggle BATCH job starts with an empty /kaggle/working, so nothing an interactive session
# produced is visible to it. Attaching a previous run's output (kernel_sources in
# kernel-metadata.json, or a dataset) makes it appear read-only under /kaggle/input. load_stage
# looks there second and copies what it finds into DATA, so a second push resumes instead of
# recomputing - without a single line of this notebook changing between pushes.
import glob as _glob

# Kaggle mounts an attached input at a path whose DEPTH depends on how it was attached: a
# notebook output lands at /kaggle/input/notebooks/<user>/<slug>/test-8-data (three levels),
# while an uploaded Dataset lands at /kaggle/input/<slug>/test-8-data (one). A single
# fixed-depth pattern silently matches neither of the others, and a silent miss here costs
# hours of recomputation - so search a BOUNDED set of depths for the folder by name, then
# fall back to any shallow input directory that simply CONTAINS the checkpoint files (which
# covers a Dataset uploaded without the enclosing folder). Bounded rather than a recursive
# walk, because /kaggle/input can hold a dataset with millions of files.
_SEED_GLOBS = [f"/kaggle/input{'/*' * _d}/test-8-data" for _d in range(0, 6)]
_SEED_FALLBACK = [f"/kaggle/input{'/*' * _d}" for _d in range(1, 3)]

def _has_checkpoints(d):
    try:
        return any(f.startswith("stage") and f.endswith(".json") for f in os.listdir(d))
    except OSError:
        return False

SEED_DIRS = []
for _pat in _SEED_GLOBS + _SEED_FALLBACK:
    for _d in sorted(_glob.glob(_pat)):
        _d = os.path.normpath(_d)
        if os.path.isdir(_d) and _d not in SEED_DIRS and _has_checkpoints(_d):
            SEED_DIRS.append(_d)
if SEED_DIRS:
    _seeded = sorted({f for d in SEED_DIRS for f in os.listdir(d) if f.endswith(".json")})
    log.info("[ckpt] seed dirs on /kaggle/input: %s", SEED_DIRS)
    log.info("[ckpt] %d checkpoint file(s) available to resume from: %s",
             len(_seeded), _seeded)
else:
    log.warning("[ckpt] NO seed dirs on /kaggle/input - every stage will be recomputed")
    if EXPECT_SEED_CHECKPOINTS:
        _mounted = (sorted(_glob.glob("/kaggle/input/*"))
                    + sorted(_glob.glob("/kaggle/input/*/*"))
                    + sorted(_glob.glob("/kaggle/input/*/*/*"))
                    + sorted(_glob.glob("/kaggle/input/*/*/*/*"))) or \
                   ["(nothing mounted at all)"]
        raise RuntimeError(f"""

==================== RESUME EXPECTED, NO CHECKPOINTS FOUND ====================
 EXPECT_SEED_CHECKPOINTS is True, so this push was meant to CONTINUE a previous
 run - but no /kaggle/input/*/test-8-data directory is mounted.

 Currently mounted under /kaggle/input (searched to depth 5 for a
 "test-8-data" folder, and depth 2 for stage*.json files directly):
   {chr(10).join('   ' + m for m in _mounted[:40])}

 Without it every stage recomputes from scratch, which costs hours and is
 discovered far too late. Halting now instead.

 FIX: in THIS notebook (inputs do not survive an import, and a fresh import has
 none), use  + Add Input -> Notebook Output -> the previous version, confirm it
 appears in the Input panel, then Save & Run All again.

 If you really do want a cold start, set EXPECT_SEED_CHECKPOINTS = False.
===============================================================================""")

def load_stage(name):
    p = _p(name)
    if not os.path.exists(p):
        for d in SEED_DIRS:                      # inherit a previous push's checkpoints
            sp = os.path.join(d, name)
            if os.path.exists(sp):
                import shutil
                shutil.copyfile(sp, p)
                log.info("[ckpt] SEEDED %s from %s", name, d)
                break
    if not os.path.exists(p): return None
    try:
        with open(p, "r", encoding="utf-8") as f: obj = json.load(f)
        log.info("[ckpt] LOAD %s (%.1f MB)", name, os.path.getsize(p) / 1e6)
        return obj
    except Exception as e:
        log.warning("[ckpt] %s unreadable (%s) - recomputing", name, e)
        return None

def save_stage(name, obj):
    """Atomic: a timeout mid-write must not leave a half file that later loads as garbage."""
    p = _p(name); fd, tmp = tempfile.mkstemp(dir=DATA, suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f: json.dump(obj, f, ensure_ascii=False)
        os.replace(tmp, p)
    finally:
        if os.path.exists(tmp): os.remove(tmp)
    log.info("[ckpt] SAVE %s (%.1f MB)", name, os.path.getsize(p) / 1e6)

# ------------------------------------------------------------------ time budget
class DeadlineReached(RuntimeError): pass

def time_left_s():
    return RUN_DEADLINE - time.time()

def deadline_hit(margin_s=DEADLINE_MARGIN_S):
    """True once the session budget is spent. Callers stop TAKING NEW WORK and flush; they
    never abandon what is already computed."""
    return time_left_s() <= margin_s

def save_run_state(stage, detail):
    """Written whenever the run stops short, so the next push knows where it stopped."""
    save_stage("run_state.json", {"stopped_in": stage, "detail": detail,
                                  "elapsed_h": round((time.time() - RUN_STARTED_AT) / 3600, 2),
                                  "budget_h": SESSION_BUDGET_H, "batch": BATCH})

# ------------------------------------------------------------------ GPU
def gpu_free_gb():
    import torch
    if not torch.cuda.is_available(): return (0.0, 0.0)
    f, t = torch.cuda.mem_get_info(); return f / 1e9, t / 1e9

def free_vram(tag=""):
    """Call AFTER `del model` at the call site."""
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    f, t = gpu_free_gb()
    log.info("VRAM released%s: free %.2f / %.2f GB", f" [{tag}]" if tag else "", f, t)

def assert_free_vram(need, label):
    import torch
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    f, t = gpu_free_gb()
    log.info("VRAM before %s: free %.2f / %.2f GB (need %.1f)", label, f, t, need)
    check(f >= need, f"free VRAM >= {need:.1f} GB before loading {label}", f"free={f:.2f}GB")

def peak_vram_gb():
    import torch
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

# ------------------------------------------------------------------ text
PRONOUN_RE = re.compile(r"(?<![A-Za-z])(he|she|him|her|his|hers|they|them|their|theirs|it|its|"
                        r"himself|herself|themselves|itself)(?![A-Za-z])", re.IGNORECASE)
# Content-word rule reused from this repo's committed injection audit: lowercase, drop
# stopwords and tokens of <=2 chars, keep hyphenated identifiers (F-1, S-II) whole.
TOKEN_RE = re.compile(r"[a-z0-9]+(?:-[a-z0-9]+)*")
STOPWORDS = set("""
a an and are as at be been being but by for from had has have he her hers him his how i if in
into is it its of on or our she that the their theirs them then there these they this those to
was were what when where which who whom why will with you your yours can could did do does done
each may might must not nor other over own same shall should some such than too very we
""".split())

def content_words(s):
    return {t for t in TOKEN_RE.findall(s.lower()) if len(t) > 2 and t not in STOPWORDS}

def pron_count(s): return len(PRONOUN_RE.findall(s))
def sha256_text(s): return hashlib.sha256(s.encode("utf-8")).hexdigest()

def pkg_versions():
    """Versions from the ACTUAL Kaggle image, stored with every stage output."""
    import importlib
    out = {"python": sys.version.split()[0]}
    for m in ["torch", "transformers", "datasets", "numpy", "pandas", "spacy",
              "fastcoref", "bitsandbytes", "accelerate", "pytrec_eval", "rank_bm25"]:
        try: out[m] = getattr(importlib.import_module(m), "__version__", "unknown")
        except Exception: out[m] = "not-installed"
    try:
        import torch
        out["cuda"] = torch.version.cuda
        out["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception: pass
    return out

ENV = pkg_versions()
log.info("ENV | %s", json.dumps(ENV))
check(str(ENV.get("transformers", "0")).split(".")[0].isdigit()
      and int(ENV["transformers"].split(".")[0]) < 5,
      "transformers pinned <5 (fastcoref/LingMess requirement)", ENV.get("transformers"))

# ------------------------------------------------------------------ PREFLIGHT (HALTING)
# Runs BEFORE Stage 0, so a too-old transformers or an unreachable/gated repo costs seconds
# instead of surfacing hours later at the first Stage 4 model load. CONFIG ONLY - no weights
# are downloaded here.
print("=" * 88)
print("PREFLIGHT - the two 2025 encoders, checked before anything expensive runs")
print("=" * 88)

_tv = tuple(int(x) for x in re.findall(r"\d+", str(ENV.get("transformers", "0")))[:2])
_needs_gemma = "embeddinggemma-300m" in EMBED_MODELS
print(f"  transformers {ENV.get('transformers')}  (need < 5 for LingMess"
      f"{'; >= 4.56 for Gemma3TextModel' if _needs_gemma else ''})")
if _needs_gemma:
    if _tv < (4, 56):
        print("  google/embeddinggemma-300m is a Gemma3TextModel and cannot load on this version.")
        print("  Fix: pip install -U 'transformers<5' in the install cell, or drop")
        print("       'embeddinggemma-300m' from EMBED_MODELS and rerun.")
    check(_tv >= (4, 56),
          "preflight: transformers >= 4.56 (google/embeddinggemma-300m needs Gemma3TextModel)",
          f"found {ENV.get('transformers')}")
else:
    print("  embeddinggemma-300m is not in the roster, so no Gemma3 version floor applies.")

from transformers import AutoConfig as _AutoConfig
PREFLIGHT_REPOS = [EMBED_MODELS[m]["hf"] for m in ("embeddinggemma-300m", "Qwen3-Embedding-0.6B")
                   if m in EMBED_MODELS]
_pf = {}
for _repo in PREFLIGHT_REPOS:
    try:
        _c = _AutoConfig.from_pretrained(_repo)          # config.json only, no weights
        _pf[_repo] = (True, f"{getattr(_c, 'model_type', '?')} | "
                            f"hidden={getattr(_c, 'hidden_size', '?')}")
    except Exception as _e:
        _pf[_repo] = (False, f"{type(_e).__name__}: {str(_e)[:220]}")
for _repo, (_ok, _detail) in _pf.items():
    print(f"  {'OK  ' if _ok else 'FAIL'}  {_repo:<32} {_detail}")
    if not _ok:
        print("  Repo unreachable. Usual causes: Kaggle internet is off; the repo is gated and")
        print("  needs a licence acceptance plus an HF token (add it as a Kaggle secret and")
        print("  huggingface_hub.login(...) before this cell); or the id is wrong.")
    check(_ok, f"preflight: {_repo} config reachable (no weights fetched)", _detail)
print("=" * 88)

## The paper's prompt — Table 5, verbatim

Transcribed from the LaTeXML HTML of arXiv:2507.07847v3, Appendix C.2. Two characters are preserved
exactly as printed and must not be "cleaned up":

* `commentary—output` uses an **em dash** (U+2014), not a hyphen.
* `Bob’s` in Example 2 uses a **right single quotation mark** (U+2019), not an ASCII apostrophe.

The fingerprint pins the string; `test-8/coref_prompt.py` in the repo holds the identical constant.

**What Table 5 does not specify, and what we chose.** It is a flat template — no system/user split,
no chat markup, no decoding parameters. Both choices below are ours, and are recorded in every stage
output so the line between *the paper's prompt* and *our run* stays visible:

1. **Role** — the whole template goes in as a single `user` turn through the model's chat template,
   because the paper shows one contiguous block.
2. **Decoding** — `do_sample=False` (greedy / temperature 0). The paper reports none; greedy is the
   only choice that makes the run reproducible.

In [ ]:
# --- Jang et al. (2025) Table 5 - VERBATIM. Do not normalise the punctuation. ---
COREF_PROMPT_TEMPLATE = (
    "You are an expert in coreference resolution. Your task is to resolve all "
    "ambiguous pronouns and references in the provided document, replacing them "
    "with explicit and contextually accurate entities. Do not add any extra text "
    "or commentary\u2014output only the fully resolved document.\n"
    "\n"
    "Below are some examples:\n"
    "\n"
    "Example 1:\n"
    "Input:\n"
    "Document: Alice, who was late, quickly ran to catch the bus because she "
    "missed her train.\n"
    "Output:\n"
    "Alice, who was late, quickly ran to catch the bus because Alice missed her "
    "train.\n"
    "\n"
    "Example 2:\n"
    "Input:\n"
    "Document: Bob said he would finish his work today because he promised his "
    "manager.\n"
    "Output:\n"
    "Bob said that Bob would finish Bob\u2019s work today because Bob promised his "
    "manager.\n"
    "\n"
    "Example 3:\n"
    "Input:\n"
    "Document: The committee stated that they would review the proposal after they "
    "received feedback.\n"
    "Output:\n"
    "The committee stated that the committee would review the proposal after the "
    "committee received feedback.\n"
    "\n"
    "When you receive the input document (which always starts with \"Document:\"), "
    "please output only the resolved document text.\n"
    "Document: {Document}"
)
PROMPT_FINGERPRINT = sha256_text(COREF_PROMPT_TEMPLATE)[:12]

def build_prompt(document):
    """`.replace`, not `.format`: a passage containing a literal brace would raise."""
    return COREF_PROMPT_TEMPLATE.replace("{Document}", document)

GENERATION_CONFIG = {"do_sample": False, "num_beams": 1, "max_new_tokens": LLM_MAX_NEW}

check(len(COREF_PROMPT_TEMPLATE) == 1069, "prompt length == 1069 chars", len(COREF_PROMPT_TEMPLATE))
check(PROMPT_FINGERPRINT == "dabdeb94b7e6", "prompt fingerprint matches coref_prompt.py",
      PROMPT_FINGERPRINT)
check("\u2014" in COREF_PROMPT_TEMPLATE, "em dash U+2014 preserved")
check("\u2019" in COREF_PROMPT_TEMPLATE, "right single quote U+2019 preserved")
print(COREF_PROMPT_TEMPLATE)

### Shared LLM helpers

Stage 0 and Stage 3 must use **identical** generation and cleaning code, otherwise Stage 0 validates
something Stage 3 does not do. Defined once here.

Outputs that come back empty, or wildly longer/shorter than the source, are recorded as **failures**
and fall back to the original passage text. They are never interpolated or silently patched, and the
failure list ships in the stage output.

In [ ]:
def load_qwen(device_index=0):
    """fp16 by default; LLM_LOAD_4BIT restores NF4 quantisation. The T4 is sm75, so bf16 is
    not usable and fp16 is the widest dtype available. fp16 is the DEFAULT here because
    bitsandbytes dequantises every weight on every forward pass: for a 3B model that already
    fits in 15 GB, NF4 buys memory the run does not need and costs throughput it does.
    `device_index` puts the whole model on one specific GPU, so Stage 3 can run an
    independent instance per device. Defaults to 0, which is what Stage 0 uses."""
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    need = 5.5 if LLM_LOAD_4BIT else LLM_VRAM_NEED
    if torch.cuda.is_available():
        _f, _t = torch.cuda.mem_get_info(device_index)
        log.info("VRAM before %s on cuda:%d: free %.2f / %.2f GB (need %.1f)",
                 LLM_MODEL, device_index, _f / 1e9, _t / 1e9, need)
        check(_f / 1e9 >= need,
              f"free VRAM >= {need:.1f} GB on cuda:{device_index} for {LLM_MODEL}",
              f"free={_f / 1e9:.2f}GB")
    else:
        assert_free_vram(need, LLM_MODEL)
    t0 = time.time()
    kw = dict(device_map={"": device_index}, torch_dtype=torch.float16,
              attn_implementation="sdpa")
    if LLM_LOAD_4BIT:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(LLM_MODEL)
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(LLM_MODEL, **kw).eval()
    log.info("%s loaded on cuda:%d in %.0fs [%s] | peak VRAM %.2f GB", LLM_MODEL, device_index,
             time.time() - t0, "4-bit nf4" if LLM_LOAD_4BIT else "fp16", peak_vram_gb())
    return tok, m

def clean_llm_output(raw):
    """Strip scaffolding the model sometimes echoes. Returns (text, was_cleaned)."""
    t0 = raw.strip(); t = t0
    for pre in ("Output:", "output:", "Document:", "Resolved document:", "Resolved Document:"):
        if t.startswith(pre): t = t[len(pre):].strip()
    if len(t) >= 2 and t[0] == t[-1] and t[0] in "\"'": t = t[1:-1].strip()
    return t, (t != t0)

def output_ok(text, source):
    if not text.strip(): return False, "empty"
    r = len(text) / max(len(source), 1)
    if r > 3.0: return False, f"too_long({r:.1f}x)"
    if r < 0.3: return False, f"too_short({r:.1f}x)"
    return True, ""

def llm_coref_batch(texts, tok, model):
    """Greedy batched generation with the paper's prompt. Returns raw decoded strings."""
    import torch
    prompts = [tok.apply_chat_template([{"role": "user", "content": build_prompt(t)}],
                                       tokenize=False, add_generation_prompt=True) for t in texts]
    enc = tok(prompts, return_tensors="pt", padding=True, truncation=True,
              max_length=3072).to(model.device)
    budget = int(min(LLM_MAX_NEW,
                     max(64, 1.6 * max(len(tok(t)["input_ids"]) for t in texts) + 64)))
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=budget, do_sample=False, num_beams=1,
                             pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

# ---------------------------------------------------------------- proper nouns
import spacy as _spacy
_NLP = None
ENT_LABELS = {"PERSON", "ORG", "GPE", "LOC", "FAC", "NORP", "EVENT", "WORK_OF_ART", "PRODUCT"}

def get_nlp():
    global _NLP
    if _NLP is None: _NLP = _spacy.load("en_core_web_sm", disable=["lemmatizer"])
    return _NLP

def proper_nouns(text):
    d = get_nlp()(text)
    out = {t.text.strip("'\u2019\".,;:()[]") for t in d if t.pos_ == "PROPN"}
    out |= {e.text.strip() for e in d.ents if e.label_ in ENT_LABELS}
    return {x for x in out if len(x) > 2}

def added_proper_nouns(src, rew):
    """Entities present in the rewrite whose surface string is absent from the source.
    Coref copies an antecedent string, so anything new here is an invention."""
    low = src.lower()
    return sorted({p for p in proper_nouns(rew) if p.lower() not in low})

## STAGE 0 · Validate the LLM coref path — **GATE**

Before spending a session's GPU budget on 8,816 passages, establish that the coref model with the
paper's prompt actually performs coreference resolution rather than paraphrase, summary, or
invention.

**This run does not use the paper's 7B model.** Jang et al. Appendix B use
`Qwen2.5-7B-Instruct`; this notebook uses `Qwen2.5-3B-Instruct` in fp16, same family, same Table 5
prompt, same chat template, so that the whole corpus can be rewritten inside one free-tier session.
That substitution is exactly what this stage exists to justify: it measures the model actually being
used, against manually written gold, before any of the expensive work happens.

**How the gate decides.** Interactively it stops and waits for a human to read the report. In a
batch job it decides from declared thresholds — the LLM must have changed something, its
generation-failure rate must be under `STAGE0_MAX_FAIL_RATE`, and its hallucinated-antecedent rate
under `STAGE0_MAX_HALLUC_RATE` — and halts if any of them is breached. The decision and the numbers
behind it are printed and written into `run_summary.json`.

This stage also decides the **Stage 3 anaphor skip**: it reports how many pronoun-free reference
chunks the model changed anyway, and the skip activates only if that count is zero.

The only labelled coref data in this repo is **test-6** (French Revolution, 204 sentence chunks),
whose rewrites were done by hand and which the repo README calls "the clean one" — all 22 of its
flagged coref-critical questions have rewrites that genuinely add entity information. The files are
fetched from a **pinned commit** and SHA-256 checked, so this validation cannot silently drift.
Tests 1–7 are read-only here and are never modified.

Reported:

* **exact-match rate** against the committed manual rewrites;
* **hallucinated antecedents** — rewrites introducing a proper noun that appears nowhere in the
  source passage. This is the failure mode that would invalidate everything downstream, because a
  hallucinated entity inside a distractor passage is a retrieval signal nobody authored.

**One thing the exact-match rate cannot mean.** test-6 chunks are single sentences, and many of its
manual rewrites resolve an antecedent living in a *different* chunk — a per-passage prompt cannot see
those, so it cannot reproduce them. The cell therefore also reports how many test-6 rewrites are
**reachable in principle** from the passage alone, which is the only honest denominator. On BEIR this
ceiling does not bind, because each passage there is a self-contained abstract.

In [ ]:
S0 = load_stage("stage0_validation.json")

if S0 is None:
    import urllib.request, torch
    from tqdm.auto import tqdm

    # ---- fetch the reference pair, verify SHA-256 ----
    t6 = {}
    for fname, want in T6_SHA.items():
        raw = urllib.request.urlopen(f"{T6_RAW}/{fname}", timeout=60).read()
        got = hashlib.sha256(raw).hexdigest()
        check(got == want, f"test-6 {fname} SHA-256 matches pinned commit", got[:16] + "...")
        t6[fname] = json.loads(raw.decode("utf-8"))

    t6_orig = {c["chunk_id"]: c["text"] for c in t6["original_chunks.json"]}
    t6_gold = {c["chunk_id"]: c["text"] for c in t6["coref_chunks.json"]}
    ids6 = sorted(t6_orig)
    check(ids6 == sorted(t6_gold), "test-6 original and coref share identical chunk ids", len(ids6))

    # ---- reachability ceiling ----
    # Words the manual rewrite introduced that were NOT already in that passage. If that
    # set is empty the antecedent was in-passage, so a per-passage prompt could in
    # principle reproduce it; otherwise the antecedent lived in a neighbouring chunk.
    changed6, reachable = [], []
    for cid in ids6:
        o, g = t6_orig[cid], t6_gold[cid]
        if o == g: continue
        changed6.append(cid)
        if not (content_words(g) - content_words(o)): reachable.append(cid)
    log.info("stage0: test-6 changed by manual coref %d/%d | reachable from passage alone %d",
             len(changed6), len(ids6), len(reachable))

    tok, llm = load_qwen()

    t0 = time.time(); preds = {}; cleaned_n = 0; failed = []
    for i in tqdm(range(0, len(ids6), LLM_BATCH), desc="stage0 llm-coref", mininterval=2.0):
        bids = ids6[i:i + LLM_BATCH]
        srcs = [t6_orig[c] for c in bids]
        for cid, src, raw in zip(bids, srcs, llm_coref_batch(srcs, tok, llm)):
            txt, was = clean_llm_output(raw); cleaned_n += int(was)
            ok, why = output_ok(txt, src)
            if not ok:
                failed.append({"chunk_id": cid, "reason": why}); txt = src
            preds[cid] = txt
    elapsed0 = time.time() - t0
    peak0 = peak_vram_gb()
    log.info("stage0: %d rewrites in %.0fs (%.2fs/passage) | cleaned %d | failed %d",
             len(preds), elapsed0, elapsed0 / max(len(preds), 1), cleaned_n, len(failed))

    del llm, tok
    free_vram("stage0")

    # ---- evidence for the Stage 3 anaphor skip (SKIP_NO_ANAPHOR) ----
    # A passage with no pronoun has nothing for the Table 5 prompt to resolve, so Stage 3 can
    # skip it and keep it verbatim - but only if THIS model really does leave such passages
    # alone. That is measurable right here, on the same 24 chunks, at no extra cost. If the
    # count below is non-zero the skip stays off and the number is reported.
    pron_free         = [c for c in ids6 if pron_count(t6_orig[c]) == 0]
    pron_free_changed = [c for c in pron_free if preds[c] != t6_orig[c]]
    log.info("stage0: pronoun-free chunks %d | changed by the LLM anyway %d",
             len(pron_free), len(pron_free_changed))

    exact       = [c for c in ids6 if preds[c] == t6_gold[c]]
    exact_reach = [c for c in reachable if preds[c] == t6_gold[c]]
    changed_pred = [c for c in ids6 if preds[c] != t6_orig[c]]

    halluc = []
    for cid in tqdm(changed_pred, desc="stage0 propn-scan", mininterval=2.0):
        add = added_proper_nouns(t6_orig[cid], preds[cid])
        if add:
            halluc.append({"chunk_id": cid, "added": add,
                           "original": t6_orig[cid], "rewrite": preds[cid]})

    po = sum(pron_count(t6_orig[c]) for c in ids6)
    pp = sum(pron_count(preds[c]) for c in ids6)
    pg = sum(pron_count(t6_gold[c]) for c in ids6)

    S0 = {
        "stage": "stage0_validation",
        "reference": f"test-6 French Revolution @ {T6_COMMIT[:8]}",
        "llm": LLM_MODEL, "paper_llm": LLM_PAPER_MODEL,
        "is_paper_model": LLM_MODEL == LLM_PAPER_MODEL,
        "quantization": ("4-bit nf4 / fp16 compute" if LLM_LOAD_4BIT
                         else "fp16, no quantisation"),
        "prompt_fingerprint": PROMPT_FINGERPRINT, "generation": GENERATION_CONFIG,
        "seed": SEED, "env": ENV,
        "n_chunks": len(ids6),
        "wall_clock_s": round(elapsed0, 1),
        "sec_per_passage": round(elapsed0 / max(len(ids6), 1), 3),
        "peak_vram_gb": round(peak0, 2),
        "manual_changed": len(changed6),
        "reachable_from_passage_alone": len(reachable),
        "llm_changed": len(changed_pred),
        "exact_match": len(exact),
        "exact_match_rate_all": round(len(exact) / len(ids6), 4),
        "exact_match_rate_reachable": round(len(exact_reach) / max(len(reachable), 1), 4),
        "hallucinated_antecedent_count": len(halluc),
        "hallucinated_rate_of_changed": round(len(halluc) / max(len(changed_pred), 1), 4),
        "hallucinated_examples": halluc[:15],
        "pronouns_original": po, "pronouns_llm": pp, "pronouns_manual_gold": pg,
        "pron_free_total": len(pron_free),
        "pron_free_changed": len(pron_free_changed),
        "pron_free_changed_ids": pron_free_changed,
        "outputs_needing_cleanup": cleaned_n, "generation_failures": failed,
        "samples": [{"chunk_id": c, "original": t6_orig[c], "manual_gold": t6_gold[c],
                     "llm": preds[c]} for c in ids6[:10]],
        "llm_rewrites": {str(c): preds[c] for c in ids6},
    }
    save_stage("stage0_validation.json", S0)
else:
    log.info("stage0: loaded from checkpoint")

In [ ]:
TOTAL_PASSAGES_EXPECTED = sum(d["expect_corpus"] for d in DATASETS.values())

print("=" * 78)
print("STAGE 0 - LLM COREF VALIDATION REPORT")
print("=" * 78)
print(f"model                    : {S0['llm']}  ({S0['quantization']})")
if not S0.get("is_paper_model", True):
    print(f"  DEPARTURE              : the paper used {S0.get('paper_llm')}. This run uses a")
    print(f"                           smaller model of the same family with the same prompt.")
    print(f"                           The numbers below are the evidence that its coref path")
    print(f"                           works; they are not a claim that it equals the 7B.")
print(f"prompt                   : Jang et al. Table 5, fingerprint {S0['prompt_fingerprint']}")
print(f"decoding                 : greedy (do_sample=False), seed {S0['seed']}")
print(f"reference                : {S0['reference']}, {S0['n_chunks']} chunks")
print("-" * 78)
print(f"manual coref changed     : {S0['manual_changed']}/{S0['n_chunks']} chunks")
print(f"  ...reachable from the passage alone : {S0['reachable_from_passage_alone']}")
print(f"LLM changed              : {S0['llm_changed']}/{S0['n_chunks']} chunks")
print(f"EXACT MATCH (all)        : {S0['exact_match']}/{S0['n_chunks']} = {S0['exact_match_rate_all']:.1%}")
print(f"EXACT MATCH (reachable)  : {S0['exact_match_rate_reachable']:.1%}   <- honest denominator")
print("-" * 78)
print(f"HALLUCINATED ANTECEDENTS : {S0['hallucinated_antecedent_count']}"
      f"  ({S0['hallucinated_rate_of_changed']:.1%} of changed passages)")
print(f"pronouns original {S0['pronouns_original']} -> llm {S0['pronouns_llm']}"
      f"   (manual gold {S0['pronouns_manual_gold']})")
print(f"outputs needing cleanup  : {S0['outputs_needing_cleanup']}")
print(f"generation failures      : {len(S0['generation_failures'])}")
print(f"wall clock               : {S0['wall_clock_s']}s ({S0['sec_per_passage']}s/passage)"
      f"  peak VRAM {S0['peak_vram_gb']} GB")
print("-" * 78)
est_h = S0["sec_per_passage"] * TOTAL_PASSAGES_EXPECTED / 3600
print(f"PROJECTED STAGE 3 COST   : {TOTAL_PASSAGES_EXPECTED:,} passages -> ~{est_h:.1f} h")
print("  NOTE: BEIR abstracts are far longer than test-6 sentences, so real cost will be")
print("        HIGHER than this. Stage 3 checkpoints every 200 passages and resumes, so it")
print("        may legitimately span two sessions.")
print("=" * 78)

if S0["hallucinated_examples"]:
    print("\nHALLUCINATED-ANTECEDENT EXAMPLES (entity in rewrite, absent from source):")
    for h in S0["hallucinated_examples"][:5]:
        print(f"\n  chunk {h['chunk_id']}  added: {h['added']}")
        print(f"    original: {h['original'][:180]}")
        print(f"    rewrite : {h['rewrite'][:180]}")

print("\nSIDE-BY-SIDE SAMPLES:")
for s in S0["samples"][:4]:
    print(f"\n  [{s['chunk_id']}] original   : {s['original'][:180]}")
    print(f"        manual gold: {s['manual_gold'][:180]}")
    print(f"        llm        : {s['llm'][:180]}")

# ---------------------------------------------------------------- anaphor-skip decision
# SKIP_NO_ANAPHOR asks for the optimisation; this is where the evidence decides it.
_pf_total   = S0.get("pron_free_total", 0)
_pf_changed = S0.get("pron_free_changed", 0)
SKIP_ACTIVE = bool(SKIP_NO_ANAPHOR and _pf_total > 0 and _pf_changed == 0)
print("-" * 78)
print(f"ANAPHOR-SKIP EVIDENCE    : {_pf_total} pronoun-free chunks, {_pf_changed} of them "
      f"changed by the LLM")
if not SKIP_NO_ANAPHOR:
    print("  skip NOT requested - every passage goes to the LLM in Stage 3")
elif _pf_total == 0:
    print("  no pronoun-free chunks in the reference, so there is NO EVIDENCE either way.")
    print("  The skip stays OFF: an untested shortcut is not worth the GPU time it saves.")
elif SKIP_ACTIVE:
    print("  the LLM left every pronoun-free chunk verbatim, so Stage 3 will skip such")
    print("  passages and keep them unchanged. The skipped count is reported in stage 3 stats.")
else:
    print(f"  the LLM CHANGED {_pf_changed} pronoun-free chunk(s): "
          f"{S0.get('pron_free_changed_ids', [])[:5]}")
    print("  the skip assumption is measurably false for this model, so it stays OFF and")
    print("  every passage goes to the LLM. This is a recorded limitation, not a silent one.")

# ---------------------------------------------------------------- the gate itself
# Interactive: STAGE0_APPROVED is False and the run stops so a human can read the report.
# Batch: STAGE0_APPROVED is None and the same judgement is made from declared thresholds.
_fail_rate = len(S0["generation_failures"]) / max(S0["n_chunks"], 1)
_hall_rate = float(S0["hallucinated_rate_of_changed"])
_gate_fail = []
if S0["llm_changed"] < STAGE0_MIN_CHANGED:
    _gate_fail.append(f"llm_changed={S0['llm_changed']} < {STAGE0_MIN_CHANGED} "
                      f"(the coref path produced no rewrites at all)")
if _fail_rate > STAGE0_MAX_FAIL_RATE:
    _gate_fail.append(f"generation-failure rate {_fail_rate:.1%} > "
                      f"{STAGE0_MAX_FAIL_RATE:.0%}")
if _hall_rate > STAGE0_MAX_HALLUC_RATE:
    _gate_fail.append(f"hallucinated-antecedent rate {_hall_rate:.1%} > "
                      f"{STAGE0_MAX_HALLUC_RATE:.0%}")

STAGE0_GATE = {"mode": "batch-auto" if STAGE0_APPROVED is None else "interactive",
               "changed": S0["llm_changed"], "fail_rate": round(_fail_rate, 4),
               "halluc_rate": round(_hall_rate, 4),
               "thresholds": {"max_fail_rate": STAGE0_MAX_FAIL_RATE,
                              "max_halluc_rate": STAGE0_MAX_HALLUC_RATE,
                              "min_changed": STAGE0_MIN_CHANGED},
               "failures": _gate_fail, "skip_no_anaphor_active": SKIP_ACTIVE}

print("\n" + "=" * 78)
print("STAGE 0 GATE")
print(f"  changed {S0['llm_changed']} (need >= {STAGE0_MIN_CHANGED}) | "
      f"failures {_fail_rate:.1%} (max {STAGE0_MAX_FAIL_RATE:.0%}) | "
      f"hallucination {_hall_rate:.1%} (max {STAGE0_MAX_HALLUC_RATE:.0%})")

if STAGE0_APPROVED is None:
    # Unattended. The gate still exists - it is decided by the thresholds above, printed
    # here, and written into run_summary.json. A run outside those bounds still halts.
    for r in _gate_fail:
        print(f"  FAIL: {r}")
    check(not _gate_fail,
          "STAGE 0 AUTO-GATE: the LLM coref path is within declared bounds",
          "; ".join(_gate_fail) or "all thresholds met")
    STAGE0_APPROVED = True
    print("  AUTO-APPROVED (batch mode) - continuing to Stage 1")
elif not STAGE0_APPROVED:
    raise RuntimeError(f"""

==================== STAGE 0 HARD GATE ====================
 Stage 0 finished and stage0_validation.json is written.
 Review the report above, then set

     STAGE0_APPROVED = True

 in the config cell and Run All again. Stage 0 will resume
 from its checkpoint - nothing is recomputed.

 (This pause is INTERACTIVE ONLY. Pushed as a Kaggle batch
  job the same decision is made automatically from the
  thresholds printed above, and the run does not stop.)
===========================================================""")
print("=" * 78)
log.info("STAGE 0 GATE: approved (%s) | anaphor-skip active=%s",
         STAGE0_GATE["mode"], SKIP_ACTIVE)

## STAGE 1 · Load full BEIR and stratify

For each dataset: pull `corpus`, `queries` and the **test** qrels, fix the query set before anything
else runs, and compute the stratification.

### Loading rules, and why each one matters

* **Passage text is `title + " " + text`**, the BEIR convention. That concatenated string is what
  gets coref-resolved and embedded.
* **All ids are cast with `str()`.** `scifact` qrels use `int64` while its corpus `_id` is a string;
  a naive join matches nothing. The gold-id assertion below would catch it, but only because of this
  normalisation.
* **Relevance grades are preserved.** `nfcorpus` is graded 0/1/2, `scifact` is binary. Flattening to
  1 would quietly mis-measure nDCG. "Gold" means `score > 0`.
* **The `queries` config holds every split.** It is filtered down to queries appearing in the test
  qrels.

### The stratification rule

> A query is `coref_critical` only if **(a)** a gold passage lacks a content word present in the
> query, **(b)** the coref rewrite adds that word, and **(c)** the word appears in **no sibling
> gold passage** for the same query.

**Why (c) is part of the rule and not a side diagnostic.** Without it the rule cannot tell
coreference from multi-hop decomposition. On HotpotQA the earlier version flagged 47 of 50 queries
when at most 4 were genuine, because in a multi-hop question each hop's gold passage necessarily
lacks the *other* hop's entity — nothing there is a pronoun to resolve. A term another gold passage
already supplies is not a coreference failure: retrieval can still succeed through that sibling.
Only terms missing from **every** gold passage count.

**Two numbers, reported separately.**

* **eligible** — conditions (a) and (c), computed here. Variant-independent, and an upper bound.
* **confirmed** — eligible *and* (b): the rewrite actually supplies the missing word. Needs a
  rewrite to exist, and the two coref variants add different tokens, so it is computed per variant
  at the Stage 2 gate and again in Stage 4.

**Everything downstream gates on `confirmed`, never on `eligible`.** Stage 1 prints the eligible
count and warns; the stratum count is measured after Stage 2, on the confirmed count, and is recorded
rather than halting.

Pronoun counts per gold passage are recorded alongside each row, but they are **not** a condition of
the rule: whether the rewrite supplies the term is the test, and a rewrite that supplies it had
something to resolve by construction.

On `scifact` (~1.1 gold/query) this barely bites. On `nfcorpus` (~38 gold/query) it is severe, and
its stratum is expected to come out near zero. That is the rule working as specified, not a bug, so
the counts are printed **with and without** the exclusion to keep the effect visible and auditable.

In [ ]:
def load_beir(name):
    """Load one BEIR dataset. Returns passages, query ids/texts, graded qrels."""
    from datasets import load_dataset
    cfg = DATASETS[name]
    corpus  = load_dataset(cfg["corpus_repo"], "corpus",  split="corpus")
    queries = load_dataset(cfg["corpus_repo"], "queries", split="queries")
    qrels_d = load_dataset(cfg["qrels_repo"],  split=cfg["split"])
    log.info("[%s] raw: corpus=%d queries=%d qrels(%s)=%d",
             name, len(corpus), len(queries), cfg["split"], len(qrels_d))

    # BEIR convention: index title + body.
    passages, n_titled = [], 0
    for r in corpus:
        title = (r.get("title") or "").strip()
        body  = (r.get("text") or "").strip()
        if title: n_titled += 1
        passages.append({"chunk_id": str(r["_id"]), "text": (title + " " + body).strip()})

    q_all = {str(r["_id"]): (r.get("text") or "").strip() for r in queries}

    # str() on BOTH sides: scifact qrels are int64, its corpus _id is a string.
    qrels = {}
    for r in qrels_d:
        qid, did, sc = str(r["query-id"]), str(r["corpus-id"]), int(r["score"])
        qrels.setdefault(qid, {})[did] = sc

    return passages, q_all, qrels, n_titled

S1 = {}
for ds in DS_NAMES:
    c_ck, q_ck, s_ck = (load_stage(f"stage1_corpus__{ds}.json"),
                        load_stage(f"stage1_queries__{ds}.json"),
                        load_stage(f"stage1_strata__{ds}.json"))
    if None in (c_ck, q_ck, s_ck):
        passages, q_all, qrels, n_titled = load_beir(ds)

        p_ids = [p["chunk_id"] for p in passages]
        check(len(set(p_ids)) == len(p_ids), f"[{ds}] corpus chunk_ids unique", len(p_ids))
        check(len(passages) == DATASETS[ds]["expect_corpus"],
              f"[{ds}] corpus size matches BEIR published size",
              f"{len(passages)} vs {DATASETS[ds]['expect_corpus']}")
        by_id = {p["chunk_id"]: p["text"] for p in passages}
        check(all(t.strip() for t in by_id.values()), f"[{ds}] no empty passage text")

        # gold = score > 0
        gold_map = {q: {d: s for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
        gold_map = {q: g for q, g in gold_map.items() if g}
        missing = sorted({d for g in gold_map.values() for d in g if d not in by_id})
        check(not missing, f"[{ds}] every qrel gold id exists in the corpus",
              f"missing={missing[:8]} n={len(missing)}")

        q_ids = sorted(q for q in gold_map if q in q_all)
        check(len(q_ids) == len(gold_map),
              f"[{ds}] every judged query id exists in the queries file",
              f"{len(q_ids)} vs {len(gold_map)}")
        check(len(q_ids) > 0, f"[{ds}] at least one judged query")
        q_texts = {q: q_all[q] for q in q_ids}
        check(all(t.strip() for t in q_texts.values()), f"[{ds}] no empty query text")

        grades = sorted({s for g in gold_map.values() for s in g.values()})
        gpq = sum(len(gold_map[q]) for q in q_ids) / len(q_ids)
        log.info("[%s] judged queries=%d | gold/query=%.2f | relevance grades=%s",
                 ds, len(q_ids), gpq, grades)

        # ---------------- stratification (sibling-gold exclusion IS the rule) -------------
        strata = {}
        n_elig = n_elig_noexcl = 0
        for q in q_ids:
            qw = content_words(q_texts[q])
            gids = sorted(gold_map[q])
            gcw = {g: content_words(by_id[g]) for g in gids}
            rows, elig, elig_noexcl = [], False, False
            for g in gids:
                miss = qw - gcw[g]
                npron = pron_count(by_id[g])   # recorded, NOT a condition of the rule
                if not miss:
                    rows.append({"gold_id": g, "missing": [], "missing_excl_sibling": [],
                                 "pronouns": npron})
                    continue
                elig_noexcl = True
                sib = set().union(*[gcw[h] for h in gids if h != g]) if len(gids) > 1 else set()
                excl = sorted(miss - sib)          # <- sibling-gold exclusion
                if excl: elig = True
                rows.append({"gold_id": g, "missing": sorted(miss),
                             "missing_excl_sibling": excl, "pronouns": npron})
            n_elig += int(elig); n_elig_noexcl += int(elig_noexcl)
            strata[q] = {"query": q_texts[q], "gold": rows, "coref_eligible": elig,
                         "coref_eligible_no_exclusion": elig_noexcl}

        # ---------------- corpus-level coref headroom -------------
        pron_tot = sum(pron_count(t) for t in by_id.values())
        with_pron = sum(1 for t in by_id.values() if pron_count(t) > 0)
        gold_ids = sorted({d for g in gold_map.values() for d in g})
        gold_with_pron = sum(1 for d in gold_ids if pron_count(by_id[d]) > 0)

        c_ck = {"stage": "stage1_corpus", "dataset": ds, "repo": DATASETS[ds]["corpus_repo"],
                "n": len(passages), "passages": passages,
                "titled_passages": n_titled, "env": ENV, "seed": SEED}
        q_ck = {"stage": "stage1_queries", "dataset": ds, "split": DATASETS[ds]["split"],
                "q_ids": q_ids, "queries": q_texts, "qrels": gold_map,
                "relevance_grades": grades, "gold_per_query": round(gpq, 2),
                "query_set_sha": sha256_text("|".join(q_ids)), "env": ENV, "seed": SEED}
        s_ck = {"stage": "stage1_strata", "dataset": ds,
                "rule": ("coref_critical = (a) a gold passage lacks a content word present in "
                         "the query, (b) the coref rewrite adds that word, and (c) the word "
                         "appears in NO sibling gold passage for the same query. Sibling "
                         "exclusion is part of the rule, not a separate diagnostic: without it "
                         "the rule cannot distinguish coreference from multi-hop decomposition. "
                         "ELIGIBLE here = (a) AND (c), variant-independent, an upper bound. "
                         "CONFIRMED = eligible AND (b), computed per variant at the Stage 2 gate "
                         "and in Stage 4. Gating is on CONFIRMED. Pronoun counts are recorded "
                         "per gold passage but are not a condition."),
                "n_queries": len(q_ids), "n_coref_eligible": n_elig,
                "n_coref_eligible_no_exclusion": n_elig_noexcl,
                "share_coref_eligible": round(n_elig / len(q_ids), 4),
                "gold_per_query": round(gpq, 2),
                "corpus_pronouns_total": pron_tot,
                "corpus_passages_with_pronoun": with_pron,
                "corpus_pronoun_share": round(with_pron / len(passages), 4),
                "gold_passages": len(gold_ids),
                "gold_with_pronoun": gold_with_pron,
                "gold_pronoun_share": round(gold_with_pron / max(len(gold_ids), 1), 4),
                "min_critical_threshold": MIN_CRITICAL,
                "strata": strata, "env": ENV, "seed": SEED}

        save_stage(f"stage1_corpus__{ds}.json",  c_ck)
        save_stage(f"stage1_queries__{ds}.json", q_ck)
        save_stage(f"stage1_strata__{ds}.json",  s_ck)
    else:
        log.info("[%s] stage1: loaded from checkpoint", ds)

    S1[ds] = {"corpus": c_ck, "queries": q_ck, "strata": s_ck}

# convenience handles
P     = {ds: S1[ds]["corpus"]["passages"] for ds in DS_NAMES}
PIDS  = {ds: [p["chunk_id"] for p in P[ds]] for ds in DS_NAMES}
QIDS  = {ds: S1[ds]["queries"]["q_ids"] for ds in DS_NAMES}
QTXT  = {ds: S1[ds]["queries"]["queries"] for ds in DS_NAMES}
QRELS = {ds: S1[ds]["queries"]["qrels"] for ds in DS_NAMES}
GOLD  = {ds: {d for g in QRELS[ds].values() for d in g} for ds in DS_NAMES}

In [ ]:
print("=" * 92)
print("STAGE 1 - BENCHMARK SUMMARY AND STRATIFICATION")
print("=" * 92)
hdr = f"{'dataset':<10} {'corpus':>7} {'queries':>8} {'gold/q':>7} {'grades':>9} " \
      f"{'pron%corp':>10} {'pron%gold':>10}"
print(hdr); print("-" * 92)
for ds in DS_NAMES:
    s, q = S1[ds]["strata"], S1[ds]["queries"]
    print(f"{ds:<10} {S1[ds]['corpus']['n']:>7,} {s['n_queries']:>8} "
          f"{s['gold_per_query']:>7.2f} {str(q['relevance_grades']):>9} "
          f"{s['corpus_pronoun_share']:>9.1%} {s['gold_pronoun_share']:>9.1%}")
print("-" * 92)
TOT_P = sum(S1[ds]["corpus"]["n"] for ds in DS_NAMES)
TOT_Q = sum(S1[ds]["strata"]["n_queries"] for ds in DS_NAMES)
# corpus totals only - no result in this notebook is pooled across datasets
print(f"{'TOTAL':<10} {TOT_P:>7,} {TOT_Q:>8}   (corpus size only; results stay per dataset)")
print()
print("STRATIFICATION (sibling-gold exclusion is part of the rule)")
print("-" * 92)
print(f"{'dataset':<10} {'eligible':>9} {'of':>5} {'share':>7}   "
      f"{'no-exclusion':>13}   effect of the exclusion")
for ds in DS_NAMES:
    s = S1[ds]["strata"]
    drop = s["n_coref_eligible_no_exclusion"] - s["n_coref_eligible"]
    print(f"{ds:<10} {s['n_coref_eligible']:>9} {s['n_queries']:>5} "
          f"{s['share_coref_eligible']:>7.1%}   {s['n_coref_eligible_no_exclusion']:>13}   "
          f"-{drop} queries excluded as sibling-supplied")
print("-" * 92)

for ds in DS_NAMES:
    s = S1[ds]["strata"]
    if s["corpus_pronoun_share"] < 0.25:
        print(f"WARNING [{ds}]: only {s['corpus_pronoun_share']:.1%} of passages contain a pronoun.")
        print("  Coref has very little to change here, so a null result would be a property of the")
        print("  corpus rather than a finding about the method. Weigh this before spending GPU.")
    if s["gold_per_query"] > 5:
        print(f"NOTE [{ds}]: {s['gold_per_query']:.0f} gold passages per query, so the sibling-gold")
        print("  exclusion is severe by construction and this stratum may legitimately be ~0.")

# ---- ELIGIBLE is reported, NOT gated on. The CONFIRMED count is measured after Stage 2:
# eligible only asks whether a query content word is missing from every gold passage, not
# whether any rewrite supplies it, and only the second half is the stratification rule.
pooled_elig = sum(S1[ds]["strata"]["n_coref_eligible"] for ds in DS_NAMES)
log.info("stage1: pooled coref-ELIGIBLE = %d across %d queries (upper bound; the recorded stratum count "
         "is on CONFIRMED, after Stage 2)", pooled_elig, TOT_Q)
print()
print(f"ELIGIBLE (upper bound, pre-rewrite): pooled {pooled_elig} / {TOT_Q} queries   "
      f"per-dataset { {d: S1[d]['strata']['n_coref_eligible'] for d in DS_NAMES} }")
print(f"CONFIRMED (post-rewrite) is computed after Stage 2 and is what the gate uses "
      f"(threshold {MIN_CRITICAL}).")
if pooled_elig < MIN_CRITICAL:
    print(f"WARNING: eligible ({pooled_elig}) is already below {MIN_CRITICAL}. Confirmed is a "
          f"subset of eligible, so the stratified arm cannot be powered and the stratum gate "
          f"after Stage 2 will halt. That emptiness is itself the finding.")

### Shared verification for both coref stages

Stages 2 and 3 must emit byte-identical schemas and pass the same checks — that is what makes the
LingMess-vs-LLM comparison *controlled* rather than confounded: identical passages, identical ids,
identical output shape, identical validation. So the checker is written once and applied to both.

The corpus-coverage check is the important one. This repo already documents a query-term-injection
problem caused by treating gold text differently from the rest. The corpus-level version of that
mistake — resolving coref only on gold passages — would invalidate the whole comparison, so gold and
non-gold counts are asserted against the full corpus and logged separately.

In [ ]:
# REQUIRED. Both coref stages must carry every one of these, which is what makes the A/B
# controlled: identical structure, identical passage payload, only the method differs.
COREF_SCHEMA_KEYS = ["stage", "dataset", "variant", "method", "n", "passages", "stats",
                     "flags", "wall_clock_s", "peak_vram_gb", "seed", "env"]

# PROVENANCE-ONLY, allow-listed by name. Stage 3 records which LLM produced its rewrites and
# whether that was the paper's model; there is no LingMess equivalent of that, so demanding
# both stages carry these keys would be meaningless. They hold no measurement, so their
# presence cannot affect a comparison. Naming them explicitly - rather than ignoring unknown
# keys - keeps the check strict: anything outside BOTH sets is still a failure.
COREF_SCHEMA_OPTIONAL = ["paper_llm", "is_paper_model"]

def verify_coref_stage(obj, ds, variant):
    tag = f"[{ds}/{variant}]"
    missing = sorted(set(COREF_SCHEMA_KEYS) - set(obj))
    unexpected = sorted(set(obj) - set(COREF_SCHEMA_KEYS) - set(COREF_SCHEMA_OPTIONAL))
    check(not missing and not unexpected, f"{tag} schema is canonical",
          f"missing={missing} unexpected={unexpected}")
    ps = obj["passages"]
    check(len(ps) == len(P[ds]), f"{tag} output length == input length",
          f"{len(ps)} vs {len(P[ds])}")
    out_ids = [p["chunk_id"] for p in ps]
    check(out_ids == PIDS[ds], f"{tag} every chunk_id preserved AND order matches input")
    check(len(set(out_ids)) == len(out_ids), f"{tag} no duplicated passages")
    check(all(set(p) == {"chunk_id", "text", "coref_text"} for p in ps),
          f"{tag} every passage has exactly chunk_id/text/coref_text")
    check(all(a["text"] == b["text"] for a, b in zip(ps, P[ds])),
          f"{tag} original text unmodified (only coref_text differs)")
    ng = sum(1 for p in ps if p["chunk_id"] in GOLD[ds])
    nn = len(ps) - ng
    check(ng == len(GOLD[ds]), f"{tag} all gold passages processed", ng)
    check(ng + nn == len(P[ds]), f"{tag} WHOLE corpus processed, not only gold",
          f"gold={ng} non_gold={nn}")
    log.info("%s coverage: gold=%d non_gold=%d total=%d", tag, ng, nn, len(ps))
    return True

def coref_stats(ps, ds, variant):
    from tqdm.auto import tqdm
    changed = [p for p in ps if p["coref_text"] != p["text"]]
    po = sum(pron_count(p["text"]) for p in ps)
    pa = sum(pron_count(p["coref_text"]) for p in ps)
    lo = float(np.mean([len(p["text"]) for p in ps]))
    la = float(np.mean([len(p["coref_text"]) for p in ps]))
    gchanged = sum(1 for p in changed if p["chunk_id"] in GOLD[ds])
    flags = []
    for p in tqdm(changed, desc=f"propn-scan[{ds}/{variant}]", mininterval=2.0):
        add = added_proper_nouns(p["text"], p["coref_text"])
        if add:
            flags.append({"chunk_id": p["chunk_id"], "added": add,
                          "is_gold": p["chunk_id"] in GOLD[ds]})
    st = {"n": len(ps), "changed": len(changed),
          "changed_share": round(len(changed) / len(ps), 4),
          "changed_gold": gchanged, "changed_non_gold": len(changed) - gchanged,
          "gold_total": len(GOLD[ds]), "non_gold_total": len(ps) - len(GOLD[ds]),
          "pronouns_before": po, "pronouns_after": pa,
          "pronoun_reduction": round(1 - pa / max(po, 1), 4),
          "mean_len_before": round(lo, 1), "mean_len_after": round(la, 1),
          "mean_len_change_pct": round(100 * (la - lo) / max(lo, 1e-9), 2),
          "added_proper_noun_passages": len(flags),
          "added_proper_noun_share_of_changed": round(len(flags) / max(len(changed), 1), 4)}
    log.info("[%s/%s] changed %d/%d (%.1f%%) | gold %d / non-gold %d | pronouns %d->%d "
             "(%.0f%% reduction) | mean len %.0f->%.0f (%+.1f%%) | added-propn %d",
             ds, variant, st["changed"], st["n"], 100 * st["changed_share"],
             st["changed_gold"], st["changed_non_gold"], po, pa,
             100 * st["pronoun_reduction"], lo, la, st["mean_len_change_pct"], len(flags))
    return st, flags

## STAGE 2 · LingMess coreference over the whole corpus

`fastcoref`'s LingMess, run **per passage**. Each BEIR passage is a self-contained abstract, so the
passage is the correct coref window and no cross-passage context is invented.

`selective=False` — resolve every cluster with a usable representative, not only proper-noun ones —
because that is what the paper's prompt instructs the LLM to do. Matching the two methods' scope is
what makes this a controlled comparison; the repo README currently lists "no controlled A/B between
LingMess and LLM coref" as an open gap, and this closes it.

In [ ]:
S2 = {}
for ds in DS_NAMES:
    ck = load_stage(f"stage2_lingmess__{ds}.json")
    if ck is None:
        import torch
        from tqdm.auto import tqdm
        # transformers 4.x + LingMess: force eager attention (same patch as test-3)
        from transformers.models.auto.modeling_auto import AutoModel
        if not getattr(AutoModel, "_coref_eager_patched", False):
            _orig = AutoModel.from_config.__func__
            @classmethod
            def _eager(cls, config, **kw):
                try: config._attn_implementation = "eager"
                except Exception: pass
                kw.setdefault("attn_implementation", "eager")
                try: return _orig(cls, config, **kw)
                except (TypeError, ValueError):
                    kw.pop("attn_implementation", None); return _orig(cls, config, **kw)
            AutoModel.from_config = _eager
            AutoModel._coref_eager_patched = True
        from fastcoref import LingMessCoref

        PRON_SET = {"he","she","him","her","his","hers","they","them","their","theirs",
                    "it","its","himself","herself","themselves","itself"}
        COMMON_CAP = {"the","a","an","this","that","these","those","if","when","you","your",
                      "it","he","she","they","we","i","in","on","at","for","to","and","but",
                      "or","there","here","what","which","who","how","why","as","is","are",
                      "was","were"}
        def _is_pron(t): return t.strip().lower() in PRON_SET
        def _has_pron(t): return any(w.strip(".,;:'\"").lower() in PRON_SET for w in t.split())
        def _has_propn(t):
            for x in t.split():
                w = x.strip(".,;:'\"()[]")
                if len(w) >= 2 and w[0].isupper() and not w.isupper() \
                   and w.lower() not in COMMON_CAP:
                    return True
            return False
        def _pick_rep(spans):
            proper = [s for s in spans if not _has_pron(s) and _has_propn(s)]
            clean  = [s for s in spans if not _has_pron(s)]
            pool = proper or clean
            return min(pool, key=len).strip() if pool else None
        def clusters_to_edits(text, clusters):
            edits = []
            for cl in clusters:
                spans = [text[s:e] for s, e in cl]
                rep = _pick_rep(spans)
                if not rep: continue
                for (s, e), st in zip(cl, spans):
                    if _is_pron(st) and st.strip().lower() != rep.lower():
                        poss = st.strip().lower() in {"his","her","its","their","hers","theirs"}
                        edits.append((s, e, rep + "'s" if poss else rep))
            return sorted(edits, key=lambda x: x[0])
        def apply_edits(text, edits):
            out = text
            for s, e, r in sorted(edits, key=lambda x: x[0], reverse=True):
                out = out[:s] + r + out[e:]
            return out

        assert_free_vram(2.0, f"LingMessCoref[{ds}]")
        t0 = time.time()
        cm = LingMessCoref(device="cuda" if torch.cuda.is_available() else "cpu")
        log.info("[%s] LingMess loaded in %.0fs", ds, time.time() - t0)

        t0 = time.time(); out_texts = {}; CH = 256
        for i in tqdm(range(0, len(P[ds]), CH), desc=f"stage2 lingmess[{ds}]", mininterval=2.0):
            batch = P[ds][i:i + CH]
            txts = [p["text"] for p in batch]
            try:
                preds = cm.predict(texts=txts)
            except Exception as e:
                log.warning("[%s] batch at %d failed (%s) - keeping originals", ds, i, e)
                for p, t in zip(batch, txts): out_texts[p["chunk_id"]] = t
                continue
            for p, t, pr in zip(batch, txts, preds):
                try:
                    out_texts[p["chunk_id"]] = apply_edits(
                        t, clusters_to_edits(t, pr.get_clusters(as_strings=False)))
                except Exception as e:
                    log.warning("[%s] passage %s failed (%s) - kept original", ds, p["chunk_id"], e)
                    out_texts[p["chunk_id"]] = t
        el = time.time() - t0; pk = peak_vram_gb()
        log.info("[%s] stage2: %d passages in %.0fs (%.3fs/passage) | peak VRAM %.2f GB",
                 ds, len(out_texts), el, el / max(len(out_texts), 1), pk)
        del cm; free_vram(f"stage2/{ds}")

        check(len(out_texts) == len(P[ds]), f"[{ds}] lingmess produced a rewrite for every passage",
              f"{len(out_texts)} vs {len(P[ds])}")
        ps2 = [{"chunk_id": p["chunk_id"], "text": p["text"],
                "coref_text": out_texts[p["chunk_id"]]} for p in P[ds]]
        st, fl = coref_stats(ps2, ds, "lingmess")
        ck = {"stage": "stage2_lingmess", "dataset": ds, "variant": "lingmess",
              "method": "fastcoref LingMessCoref, per-passage, selective=False",
              "n": len(ps2), "passages": ps2, "stats": st, "flags": fl,
              "wall_clock_s": round(el, 1), "peak_vram_gb": round(pk, 2),
              "seed": SEED, "env": ENV}
        save_stage(f"stage2_lingmess__{ds}.json", ck)
    else:
        log.info("[%s] stage2: loaded from checkpoint", ds)
    verify_coref_stage(ck, ds, "lingmess")
    S2[ds] = ck

for ds in DS_NAMES:
    print(f"\n--- {ds} / lingmess ---")
    print(json.dumps(S2[ds]["stats"], indent=2))

### Confirmed coref-critical stratum — measured before the expensive stage, never halting

The Stage 1 eligible count is only an **upper bound**: it asks whether a gold passage is missing a
query content word and contains a pronoun, not whether any rewrite actually *supplies* that word.
That second half is what the stratification rule really turns on, and it needs a rewrite to exist.

LingMess costs minutes. Stage 3 costs hours. So the confirmed count is computed here, from the
Stage 2 rewrites, and the power check is applied **before** committing the GPU budget.

If the stratum is too thin, that emptiness is itself a finding — it would explain both this repo's
null results and the paper's very small numbers, because an effect confined to a handful of queries
cannot survive being averaged over hundreds. The **aggregate arm** (paper replication, pooling
contrast, confidence intervals) does not depend on this stratum and stays valid either way.

So a thin stratum is **recorded, not fatal**: the count is printed here, carried into
`stage4_run_meta.json` and `run_summary.json`, and reported by the analysis as a diagnostic. The
run continues. What would actually be dishonest is publishing a stratified result computed on a
stratum this small — and that is prevented in the analysis, which marks any scope below `MIN_FIT`
as underpowered rather than fitting it.

In [ ]:
def confirm_critical(variant_text_by_id, ds, variant):
    """CONFIRMED = eligible AND the rewrite actually adds one of the query content words the
    gold passage was missing (after sibling-gold exclusion). This is the number every gate and
    every stratified table uses; `eligible` is only its upper bound."""
    orig = {p["chunk_id"]: p["text"] for p in P[ds]}
    flags, evidence = {}, {}
    for q in QIDS[ds]:
        rec = S1[ds]["strata"]["strata"][q]
        crit, why = False, []
        for g in rec["gold"]:
            need = set(g["missing_excl_sibling"])
            if not need: continue      # pronoun count is recorded, but is not a condition
            gid = g["gold_id"]
            added = content_words(variant_text_by_id[gid]) - content_words(orig[gid])
            got = sorted(need & added)
            if got:
                crit = True; why.append({"gold_id": gid, "supplied": got})
        flags[q] = crit
        if crit: evidence[q] = why
    n = sum(flags.values())
    log.info("[%s/%s] confirmed coref-critical: %d/%d (eligible upper bound %d)",
             ds, variant, n, len(QIDS[ds]), S1[ds]["strata"]["n_coref_eligible"])
    return flags, evidence, n

CRIT_LM, EV_LM, NCRIT_LM = {}, {}, {}
for ds in DS_NAMES:
    CRIT_LM[ds], EV_LM[ds], NCRIT_LM[ds] = confirm_critical(
        {p["chunk_id"]: p["coref_text"] for p in S2[ds]["passages"]}, ds, "lingmess")

pooled_crit = sum(NCRIT_LM.values())
print("=" * 88)
print("SECOND GATE - CONFIRMED COREF-CRITICAL STRATUM (from LingMess rewrites)")
print("=" * 88)
print(f"{'dataset':<10} {'eligible':>9} {'confirmed':>10} {'of queries':>11}")
for ds in DS_NAMES:
    print(f"{ds:<10} {S1[ds]['strata']['n_coref_eligible']:>9} {NCRIT_LM[ds]:>10} "
          f"{len(QIDS[ds]):>11}")
# TOTAL is used ONLY by this gate, to decide whether the expensive stage is worth running.
# No reported result is pooled across datasets; the analysis is strictly per dataset.
print(f"{'TOTAL':<10} {sum(S1[d]['strata']['n_coref_eligible'] for d in DS_NAMES):>9} "
      f"{pooled_crit:>10} {sum(len(QIDS[d]) for d in DS_NAMES):>11}")
print(f"gate threshold (summed across datasets, gate only): {MIN_CRITICAL}")
print("=" * 88)
for ds in DS_NAMES:
    for q, why in list(EV_LM[ds].items())[:4]:
        print(f"  [{ds}] {QTXT[ds][q][:80]}")
        for w in why: print(f"       gold {w['gold_id']}: rewrite supplied {w['supplied']}")

# ---------------------------------------------------------------- the gate: record, do not halt
# This used to raise and wait for a human to set STRATUM_GATE_OVERRIDE. It no longer does,
# and dropping it costs nothing evidentially: a thin stratum is a FINDING about this
# benchmark, not an error in the run, and the AGGREGATE arm - the paper's own condition,
# the pooling contrast, the bootstrap CIs - does not depend on the stratum at all. The
# number is recorded here, carried into stage4_run_meta.json, and reported by the analysis
# as a diagnostic. What would be dishonest is reporting a stratified result computed on a
# stratum this small; the analysis refuses to do that, which is the real protection.
STRATUM_GATE = {"pooled_confirmed": pooled_crit, "min_required": MIN_CRITICAL,
                "per_dataset": {d: NCRIT_LM[d] for d in DS_NAMES},
                "underpowered": bool(pooled_crit < MIN_CRITICAL)}
if STRATUM_GATE["underpowered"]:
    print(f"""
==================== STRATUM: UNDERPOWERED (recorded, not fatal) ====================
 Pooled confirmed coref-critical queries: {pooled_crit} (reporting threshold {MIN_CRITICAL}).
 Per dataset: { {d: NCRIT_LM[d] for d in DS_NAMES} }

 The STRATIFIED arm is underpowered on this benchmark. That emptiness is itself a
 finding and is recorded as one. Expected for lingmess in particular: the rule needs the
 rewrite to ADD a query content word, and an extractive system can only copy strings that
 are already in the passage, so its confirmed stratum is 0 by construction.

 The run CONTINUES. The aggregate arm does not depend on this stratum.
====================================================================================""")
    log.warning("stratum underpowered: pooled=%d < %d - recorded as a finding, run continues",
                pooled_crit, MIN_CRITICAL)
else:
    log.info("stratum gate: pooled=%d >= %d", pooled_crit, MIN_CRITICAL)

## STAGE 3 · LLM coreference over the whole corpus — checkpointed every 200 passages

`Qwen2.5-3B-Instruct`, fp16, greedy, the paper's Table 5 prompt (Jang et al. used the 7B —
see the Stage 0 validation for why the smaller model is defensible here), applied per
passage across **both corpora in full** — never only the gold passages.

This is the long stage. It writes `stage3_llm_partial__{dataset}.json` every 200 passages and
resumes from it, so a session timeout costs at most 200 passages of work and the stage may
legitimately span two sessions. A live rate estimate prints at every checkpoint so you can project
the finish time early rather than discovering it at hour six.

In [ ]:
S3 = {}
for ds in DS_NAMES:
    ck = load_stage(f"stage3_llm__{ds}.json")
    if ck is None:
        import torch, threading
        from concurrent.futures import ThreadPoolExecutor
        from tqdm.auto import tqdm
        part = load_stage(f"stage3_llm_partial__{ds}.json") or \
               {"done": {}, "failed": [], "cleaned": 0, "elapsed_s": 0.0, "skipped": 0}
        done, failed = part["done"], part["failed"]
        cleaned = [part["cleaned"]]          # boxed: mutated from worker threads
        prior = part["elapsed_s"]

        # ---- anaphor skip: decided by Stage 0 evidence, not assumed (see SKIP_ACTIVE) ----
        # A passage with no pronoun has nothing for the Table 5 prompt to resolve, so it is
        # kept VERBATIM and never reaches the GPU. Skipped passages are still present in the
        # llm variant, identical to their originals, so the corpus stays complete and the
        # A/B against lingmess stays aligned id-for-id.
        _remaining = [p for p in P[ds] if p["chunk_id"] not in done]
        skipped = [p for p in _remaining if pron_count(p["text"]) == 0] if SKIP_ACTIVE else []
        for p in skipped:
            done[p["chunk_id"]] = p["text"]
        n_skipped = len(skipped)
        skipped_total = part.get("skipped", 0) + n_skipped
        todo = [p for p in _remaining if p["chunk_id"] not in done]
        n_done0 = len(done)
        log.info("[%s] stage3: %d already done, %d skipped now (no anaphor, active=%s), "
                 "%d to the LLM", ds, len(done) - n_skipped, n_skipped, SKIP_ACTIVE, len(todo))

        if todo:
            # ---------------- one Qwen instance per visible GPU ----------------
            N_GPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
            engines = []
            for gi in range(max(N_GPU, 1)):
                try:
                    engines.append((gi,) + load_qwen(device_index=gi if N_GPU else 0))
                except Exception as e:
                    log.warning("[%s] Qwen load on cuda:%d failed (%s)", ds, gi, e)
                    if not engines:
                        raise
                    break

            # ---------------- length-aware batching, shared work queue ----------------
            # What went wrong before: batch size was a per-worker ratchet that only ever went
            # DOWN, so one OOM on a long passage pinned that GPU at batch 1 for every short
            # passage afterwards. Here the batch is computed from the passages themselves -
            # a token budget divided by their estimated length - so it shrinks for long text
            # and recovers for short. The budget itself steps down on OOM (never below a
            # floor) and that is the only sticky state.
            # The queue is shared: whichever GPU is free takes the next slice, so a slower
            # device simply does fewer passages instead of becoming a straggler while the
            # other idles.
            TOKEN_BUDGET = [4500]        # ~positions per forward pass, shared, adaptive
            BUDGET_FLOOR = 1200
            PROMPT_TOKENS = 330          # Table 5 template + chat markup, measured once below
            todo_sorted = sorted(todo, key=lambda p: len(p["text"]))
            _cursor = [0]
            _dl_flag = [False]               # one warning, not one per worker
            _qlk = threading.Lock()

            def _est_tokens(p):
                return len(p["text"]) // 4 + PROMPT_TOKENS      # ~4 chars/token English

            def _next_slice():
                """Pop the next length-appropriate slice off the shared queue."""
                with _qlk:
                    i = _cursor[0]
                    if i >= len(todo_sorted):
                        return []
                    # Session budget: stop TAKING work, let in-flight batches finish, then
                    # the flush below persists everything computed so far.
                    if deadline_hit():
                        if not _dl_flag[0]:
                            _dl_flag[0] = True
                            log.warning("[%s] session budget spent at %d/%d - draining workers",
                                        ds, i, len(todo_sorted))
                        return []
                    budget = TOKEN_BUDGET[0]
                    bs = max(1, min(LLM_BATCH, budget // max(_est_tokens(todo_sorted[i]), 1)))
                    # the slice is length-sorted, so check its LONGEST member fits too
                    while bs > 1 and _est_tokens(todo_sorted[min(i + bs - 1,
                                                                len(todo_sorted) - 1)]) * bs > budget:
                        bs -= 1
                    batch = todo_sorted[i:i + bs]
                    _cursor[0] = i + len(batch)
                    return batch

            log.info("[%s] stage3 on %d GPU(s) | token budget %d | batch cap %d | "
                     "passage chars %d..%d", ds, len(engines), TOKEN_BUDGET[0], LLM_BATCH,
                     len(todo_sorted[0]["text"]), len(todo_sorted[-1]["text"]))

            t0 = time.time()
            _lk = threading.Lock()          # guards done/failed/cleaned/since
            _save_lk = threading.Lock()     # guards the checkpoint file
            state = {"since": 0}
            pbar = tqdm(total=len(todo), desc=f"stage3 llm[{ds}]x{len(engines)}", mininterval=2.0)

            def _flush(elapsed_now):
                with _lk:
                    snap = {"done": dict(done), "failed": list(failed),
                            "cleaned": cleaned[0], "elapsed_s": elapsed_now,
                            "skipped": skipped_total}
                with _save_lk:
                    save_stage(f"stage3_llm_partial__{ds}.json", snap)

            def _worker(gi, tok, llm):
                if torch.cuda.is_available():
                    torch.cuda.set_device(gi)      # thread-local: makes empty_cache() correct
                while True:
                    batch = _next_slice()
                    if not batch:
                        return
                    srcs = [p["text"] for p in batch]
                    try:
                        raws = llm_coref_batch(srcs, tok, llm)
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        with _qlk:                       # shrink the budget for everyone
                            TOKEN_BUDGET[0] = max(BUDGET_FLOOR, int(TOKEN_BUDGET[0] * 0.75))
                            nb = TOKEN_BUDGET[0]
                        log.warning("[%s/cuda:%d] OOM on %d passages - token budget -> %d, "
                                    "splitting this slice", ds, gi, len(batch), nb)
                        raws, sub = [], max(1, len(batch) // 2)
                        for j in range(0, len(batch), sub):
                            part_srcs = srcs[j:j + sub]
                            try:
                                raws += llm_coref_batch(part_srcs, tok, llm)
                            except Exception as e:
                                log.warning("[%s/cuda:%d] sub-batch of %d failed (%s) - kept "
                                            "as original", ds, gi, len(part_srcs), e)
                                torch.cuda.empty_cache()
                                raws += [""] * len(part_srcs)
                    rows = []
                    for p, src, raw in zip(batch, srcs, raws):
                        txt, was = clean_llm_output(raw)
                        ok, why = output_ok(txt, src)
                        rows.append((p["chunk_id"], src if not ok else txt,
                                     None if ok else why, was))
                    with _lk:
                        for cid, txt, why, was in rows:
                            cleaned[0] += int(was)
                            if why:
                                failed.append({"chunk_id": cid, "reason": why})
                            done[cid] = txt
                        state["since"] += len(batch)
                        need = state["since"] >= LLM_CKPT_EVERY
                        if need:
                            state["since"] = 0
                        n_now = len(done)
                    pbar.update(len(batch))
                    if need:
                        _flush(prior + (time.time() - t0))
                        rate = (time.time() - t0) / max(n_now - n_done0, 1)
                        left = (len(P[ds]) - n_now) * rate / 3600
                        log.info("[%s] ckpt %d/%d | %.2fs/passage wall | ~%.1fh remaining | "
                                 "budget %d", ds, n_now, len(P[ds]), rate, left,
                                 TOKEN_BUDGET[0])

            with ThreadPoolExecutor(max_workers=len(engines)) as ex:
                futs = [ex.submit(_worker, gi, tok, llm) for gi, tok, llm in engines]
                for f_ in futs:
                    f_.result()          # re-raises any worker exception here
            pbar.close()

            el = prior + (time.time() - t0); pk = peak_vram_gb()
            _flush(el)
            log.info("[%s] stage3 finished in %.0fs | final token budget %d",
                     ds, time.time() - t0, TOKEN_BUDGET[0])
            for gi, tok, llm in engines:
                del tok, llm
            del engines
            free_vram(f"stage3/{ds}")
        else:
            el, pk = prior, 0.0

        if len(done) < len(P[ds]):
            # The session budget ran out mid-dataset. Everything computed IS on disk - the
            # partial checkpoint above is authoritative and the next push resumes from it.
            # The run stops here rather than continuing, because Stage 4 on a half-rewritten
            # corpus would silently mix rewritten and original passages inside one variant.
            _miss = len(P[ds]) - len(done)
            save_run_state("stage3", {"dataset": ds, "done": len(done), "missing": _miss,
                                      "skipped_total": skipped_total})
            print("=" * 88)
            print("SESSION BUDGET REACHED DURING STAGE 3 - NOT A FAILURE")
            print("=" * 88)
            print(f"  dataset        : {ds}")
            print(f"  rewritten      : {len(done)} / {len(P[ds])}   ({_miss} left)")
            print(f"  elapsed        : {(time.time() - RUN_STARTED_AT) / 3600:.2f} h of "
                  f"{SESSION_BUDGET_H:.1f} h budget")
            print("  Everything computed is checkpointed in stage3_llm_partial__"
                  f"{ds}.json.")
            print("  TO RESUME: save this kernel's output, add it to kernel-metadata.json as a")
            print("  kernel_sources entry, and push again. load_stage() picks the checkpoints")
            print("  up from /kaggle/input automatically - no code changes.")
            print("=" * 88)
            raise DeadlineReached(
                f"stage3/{ds}: {len(done)}/{len(P[ds])} rewritten when the "
                f"{SESSION_BUDGET_H:.1f}h budget expired - resume by attaching this output")
        check(len(done) == len(P[ds]), f"[{ds}] stage3 produced a rewrite for every passage",
              f"{len(done)} vs {len(P[ds])}")
        ps3 = [{"chunk_id": p["chunk_id"], "text": p["text"],
                "coref_text": done[p["chunk_id"]]} for p in P[ds]]
        st, fl = coref_stats(ps3, ds, "llm")
        st["outputs_needing_cleanup"] = cleaned[0]
        st["llm_skipped_no_anaphor"] = skipped_total
        st["llm_skip_rule_active"] = bool(SKIP_ACTIVE)
        st["llm_sent_to_model"] = len(P[ds]) - skipped_total
        st["generation_failures"] = len(failed)
        st["failure_detail"] = failed[:200]
        ck = {"stage": "stage3_llm", "dataset": ds, "variant": "llm",
              "method": (f"{LLM_MODEL} "
                         f"{'4-bit nf4/fp16' if LLM_LOAD_4BIT else 'fp16'}, greedy, "
                         f"Jang et al. Table 5 prompt (fingerprint {PROMPT_FINGERPRINT}), "
                         f"per-passage"
                         + (f"; {skipped_total} pronoun-free passages kept verbatim without "
                            f"an LLM call" if SKIP_ACTIVE else "")),
              "paper_llm": LLM_PAPER_MODEL, "is_paper_model": LLM_MODEL == LLM_PAPER_MODEL,
              "n": len(ps3), "passages": ps3, "stats": st, "flags": fl,
              "wall_clock_s": round(el, 1), "peak_vram_gb": round(pk, 2),
              "seed": SEED, "env": ENV}
        save_stage(f"stage3_llm__{ds}.json", ck)
    else:
        log.info("[%s] stage3: loaded from checkpoint", ds)
    verify_coref_stage(ck, ds, "llm")
    S3[ds] = ck

# Both coref stages must emit the same REQUIRED schema - this is what makes the A/B
# controlled. They may differ in the allow-listed provenance keys (COREF_SCHEMA_OPTIONAL),
# which carry no measurement; an undeclared key on either side is still a failure.
for ds in DS_NAMES:
    _req, _opt = set(COREF_SCHEMA_KEYS), set(COREF_SCHEMA_OPTIONAL)
    check(_req <= set(S2[ds]) and _req <= set(S3[ds]),
          f"[{ds}] stage 2 and stage 3 both carry the full required schema",
          f"s2_missing={sorted(_req - set(S2[ds]))} s3_missing={sorted(_req - set(S3[ds]))}")
    check(not ((set(S2[ds]) | set(S3[ds])) - _req - _opt),
          f"[{ds}] neither coref stage carries an undeclared top-level key",
          sorted((set(S2[ds]) | set(S3[ds])) - _req - _opt))
    check(set(S2[ds]["stats"]) <= set(S3[ds]["stats"]),
          f"[{ds}] stage 3 stats is a superset of stage 2 stats",
          sorted(set(S2[ds]["stats"]) - set(S3[ds]["stats"])))
    check([p["chunk_id"] for p in S2[ds]["passages"]] == [p["chunk_id"] for p in S3[ds]["passages"]],
          f"[{ds}] both coref stages cover identical ids in identical order")
    print(f"\n--- {ds} / llm ---")
    print(json.dumps({k: v for k, v in S3[ds]["stats"].items() if k != "failure_detail"}, indent=2))

## STAGE 4 · The GPU run — writes raw per-query rows, computes no findings

**The run and the analysis are separated.** This stage does one thing: encode, retrieve, score, and
write **one row per (model, dataset, condition, retrieval, query)** to
`test-8-data/per_query_results.jsonl`. It produces no tables and no plots. Every finding in this
notebook is computed in the final cell, from that file alone, with no GPU.

Why: a Kaggle session that dies at model 4 must not cost models 1–3. Rows are appended **after each
model finishes**, then `flush()` + `os.fsync()`, and only then is the model recorded in
`stage4_done_models.json`. On restart, rows belonging to any model not in that list — plus a torn
final line, if the kernel was killed mid-write — are pruned, and the run resumes at the next model.
The completed models' rows are already analysable.

### Row schema

| field | meaning |
|---|---|
| `model`, `model_role`, `model_year`, `pooling`, `prefix_scheme` | which encoder produced the row, and how it was prompted |
| `dataset` | `scifact` or `nfcorpus` — **never pooled** |
| `condition` | `baseline` (original corpus) or `coref` |
| `coref_method` | `none`, `lingmess`, or `llm` |
| `retrieval` | `dense`, or `hybrid` (0.7 dense + 0.3 BM25 over the original text) |
| `qid` | BEIR query id |
| `top50` | top-50 retrieved **doc ids**, so any other metric can be recomputed later without a GPU |
| `recall_10`, `ndcg_10`, `mrr` | this query's scores, graded qrels |
| `in_confirmed_stratum` | the coref rewrite supplied a query content word that no gold passage had |
| `gold_has_pronoun` | at least one gold passage contains a pronoun in the original text |
| `n_gold` | number of gold passages for this query |

A `baseline` row is shared by both coref methods, so its `in_confirmed_stratum` is the union across
methods; the analysis always filters on the **coref** row's flag, never the baseline's.

### Model order, dtype, and the sanity floor

Run order is **cheapest first** — MiniLM → bge-small → e5-small → Qwen3-0.6B — so a
misconfiguration surfaces in minutes rather than hours, and the three small encoders are on disk
before the expensive one begins.

| model | pooling | dtype | prefix scheme |
|---|---|---|---|
| `sentence-transformers/all-MiniLM-L6-v2` | mean | fp16 | `none` |
| `BAAI/bge-small-en-v1.5` | CLS | fp16 | `bge_query_instruction` |
| `intfloat/e5-small-v2` | mean | fp16 | `e5_query_passage` |
| `Qwen/Qwen3-Embedding-0.6B` | last-token | fp16 | `qwen3_instruct_query` |

`google/embeddinggemma-300m` was **dropped for GPU cost** — it ran fp32 (Gemma3 can overflow fp16 to
NaN) and was about a third of this stage's time. Pooled vectors are cast to fp32 before normalising
in every case.

**Sanity floor, on the ORIGINAL (non-coref) corpus, dense retrieval:** `Recall@10 ≥ 0.50` on
`scifact`, `nDCG@10 ≥ 0.25` on `nfcorpus`. Below the floor is a misconfiguration, not a result. The
model, its `prefix_scheme`, both literal prefixes, the pooling and the dtype are printed — and the
model is **quarantined rather than fatal**: its rows are still written, flagged `floor_pass=false`,
and excluded from every table in the analysis, while the remaining models still run. In an
unattended job, halting here would throw away every model that had not started yet in order to
punish one that was misconfigured. (nfcorpus is floored on nDCG@10 because ~38 gold passages per
query cap its Recall@10 near 10/38.)

**After the first model of a session finishes**, a wall-clock projection for the whole roster
prints, scaled from that model's measured time by the declared `cost_weight`s, so the 12h session
limit can be planned for rather than discovered. The run also stops taking new models once the
session budget is spent, leaving every completed model fully analysable.

Pooling is applied **explicitly in code** rather than delegated to `sentence-transformers`, because
pooling is one of the variables under test and must be visible and auditable.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel as HFAutoModel
from tqdm.auto import tqdm

CORPUS_TEXT, VAR_IDS = {}, {}
for ds in DS_NAMES:
    CORPUS_TEXT[ds] = {"original": [p["text"] for p in P[ds]],
                       "lingmess": [p["coref_text"] for p in S2[ds]["passages"]],
                       "llm":      [p["coref_text"] for p in S3[ds]["passages"]]}
    VAR_IDS[ds] = {"original": PIDS[ds],
                   "lingmess": [p["chunk_id"] for p in S2[ds]["passages"]],
                   "llm":      [p["chunk_id"] for p in S3[ds]["passages"]]}
    for v in VARIANTS:
        check(len(CORPUS_TEXT[ds][v]) == len(P[ds]),
              f"[{ds}] variant '{v}' has the same passage count", len(CORPUS_TEXT[ds][v]))
        check(VAR_IDS[ds][v] == PIDS[ds],
              f"[{ds}] variant '{v}' has identical passage ids in identical order")
    check(sha256_text("|".join(QIDS[ds])) == S1[ds]["queries"]["query_set_sha"],
          f"[{ds}] query set unchanged since stage 1")

POOLINGS = {"cls", "mean", "last"}
check(all(c["pooling"] in POOLINGS for c in EMBED_MODELS.values()),
      "every embedding model declares a known pooling strategy",
      {m: c["pooling"] for m, c in EMBED_MODELS.items()})
check(all(c["dtype"] in {"fp16", "fp32"} for c in EMBED_MODELS.values()),
      "every embedding model declares fp16 or fp32",
      {m: c["dtype"] for m, c in EMBED_MODELS.items()})

def _pool(h, mask, how):
    if how == "cls":
        return h[:, 0]
    if how == "last":
        # last NON-PAD token. Valid only with right-side padding, which load_encoder asserts.
        idx = (mask.sum(1) - 1).clamp(min=0)
        return h[torch.arange(h.shape[0], device=h.device), idx]
    m = mask.unsqueeze(-1).to(h.dtype)
    return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

def load_encoder(mname, cfg):
    """One encoder at a time. Each model declares its own dtype: fp16 on the T4 for the
    transformer encoders, fp32 for any model whose activations can overflow fp16 to NaN.
    Logs the prefix scheme actually applied, per model."""
    assert_free_vram(cfg.get("vram_need", 1.0), cfg["hf"])
    t_load = time.time()
    want_fp16 = cfg["dtype"] == "fp16" and torch.cuda.is_available()
    dtype = torch.float16 if want_fp16 else torch.float32
    tok = AutoTokenizer.from_pretrained(cfg["hf"])
    if cfg["pooling"] == "last":
        tok.padding_side = "right"                    # _pool('last') indexes mask.sum()-1
        check(tok.padding_side == "right",
              f"{mname}: right-side padding for last-token pooling", tok.padding_side)
        check(cfg.get("append_eos") and tok.eos_token is not None,
              f"{mname}: EOS available and appended for last-token pooling", tok.eos_token)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = HFAutoModel.from_pretrained(cfg["hf"], torch_dtype=dtype).to(
        "cuda" if torch.cuda.is_available() else "cpu").eval()
    log.info("stage4: %s loaded in %.0fs | pooling=%s | dtype=%s | %dM params (%d) | "
             "role=%s | batch=%d", mname, time.time() - t_load, cfg["pooling"], dtype,
             cfg["params_m"], cfg["year"], cfg["role"], cfg.get("batch", EMBED_BATCH))
    log.info("stage4: %s PREFIX SCHEME=%s | query=%r | passage=%r | append_eos=%s",
             mname, cfg["prefix_scheme"], cfg["query_prefix"], cfg["passage_prefix"],
             bool(cfg.get("append_eos")))
    return tok, mdl

@torch.no_grad()
def encode(texts, tok, mdl, cfg, prefix, desc):
    """`prefix` is the model's own documented query- or passage-side convention.
    Pooled vectors are cast to fp32 BEFORE normalising, so an fp16 forward pass cannot
    put a denormal or an overflow into the similarity matrix."""
    bs = cfg.get("batch", EMBED_BATCH)
    eos = tok.eos_token if (cfg.get("append_eos") and tok.eos_token) else ""
    vecs = []
    for i in tqdm(range(0, len(texts), bs), desc=desc, mininterval=2.0):
        batch = [prefix + t + eos for t in texts[i:i + bs]]
        enc = tok(batch, padding=True, truncation=True,
                  max_length=MAX_LEN, return_tensors="pt").to(mdl.device)
        v = _pool(mdl(**enc).last_hidden_state, enc["attention_mask"], cfg["pooling"]).float()
        vecs.append(torch.nn.functional.normalize(v, p=2, dim=1).cpu().numpy())
    return np.vstack(vecs)

def assert_emb_sane(M, label):
    check(np.isfinite(M).all(), f"{label}: embeddings finite (no NaN/Inf)", f"shape={M.shape}")
    n = np.linalg.norm(M, axis=1)
    check((n > 1e-6).all(), f"{label}: no all-zero vectors", f"min_norm={n.min():.2e}")

def dense_ranklist(qv, pv, depth=RUN_DEPTH):
    sims = qv @ pv.T
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[i, j])) for j in order[i]] for i in range(sims.shape[0])]

def weighted_rrf(d_rl, b_rl, dense_w=DENSE_WEIGHT, k=RRF_K, depth=RUN_DEPTH):
    fused = []
    for qi in range(len(d_rl)):
        acc = {}
        for w, rl in ((dense_w, d_rl), (1.0 - dense_w, b_rl)):
            for rank, (pidx, _s) in enumerate(rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + w / (k + rank + 1)
        fused.append([(int(p), float(s))
                      for p, s in sorted(acc.items(), key=lambda x: -x[1])[:depth]])
    return fused

def top_doc_ids(rl_row, ds, n=SAVE_TOPN):
    """Corpus indices -> BEIR doc ids, top-n, for the raw per-query row."""
    return [PIDS[ds][p] for p, _s in rl_row[:n]]

# ---------------- BM25 over ORIGINAL text, per dataset ----------------
from rank_bm25 import BM25Okapi
_btok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
BM25_RL = {}
for ds in DS_NAMES:
    log.info("[%s] building BM25 over ORIGINAL text (%d passages)", ds, len(P[ds]))
    bm = BM25Okapi([_btok(t) for t in CORPUS_TEXT[ds]["original"]])
    rl = []
    for q in tqdm(QIDS[ds], desc=f"bm25[{ds}]", mininterval=2.0):
        sc = bm.get_scores(_btok(QTXT[ds][q]))
        order = np.argsort(-sc)[:RUN_DEPTH]
        rl.append([(int(j), float(sc[j])) for j in order])
    BM25_RL[ds] = rl
    del bm

In [ ]:
import pytrec_eval

# ---------------------------------------------------------------- per-query metrics
def per_query_metrics(ds, rl):
    """pytrec_eval with GRADED qrels (nfcorpus is 0/1/2). One value per query, in QIDS order."""
    qrels_ev = {q: {d: int(s) for d, s in QRELS[ds][q].items()} for q in QIDS[ds]}
    run = {qid: {PIDS[ds][p]: float(s) for p, s in rl[i]} for i, qid in enumerate(QIDS[ds])}
    per = pytrec_eval.RelevanceEvaluator(qrels_ev, MEASURES).evaluate(run)
    return {m: np.array([per[q][m] for q in QIDS[ds]], dtype=float) for m in METRIC_NAMES}

# ---------------------------------------------------------------- query-level facts
# Confirmed stratum, per dataset per coref variant (the rule's second half: the rewrite
# actually supplies a query content word that no gold passage had).
CRIT, EV, NCRIT = {}, {}, {}
for ds in DS_NAMES:
    CRIT[ds], EV[ds], NCRIT[ds] = {}, {}, {}
    for v in COREF_VARIANTS:
        CRIT[ds][v], EV[ds][v], NCRIT[ds][v] = confirm_critical(
            dict(zip(PIDS[ds], CORPUS_TEXT[ds][v])), ds, v)

# gold_has_pronoun: does ANY gold passage for this query contain a pronoun in the ORIGINAL
# text. This is the subset where coreference resolution has anything at all to act on.
GOLD_HAS_PRON, N_GOLD = {}, {}
for ds in DS_NAMES:
    GOLD_HAS_PRON[ds] = {q: bool(any(g["pronouns"] > 0
                                     for g in S1[ds]["strata"]["strata"][q]["gold"]))
                         for q in QIDS[ds]}
    N_GOLD[ds] = {q: len(QRELS[ds][q]) for q in QIDS[ds]}
    log.info("[%s] gold_has_pronoun: %d/%d queries | confirmed stratum: %s",
             ds, sum(GOLD_HAS_PRON[ds].values()), len(QIDS[ds]),
             {v: NCRIT[ds][v] for v in COREF_VARIANTS})

# ---------------------------------------------------------------- stratum preflight
# ADVISORY, and it runs BEFORE any encoding, so the size of the stratified arm is known
# before the GPU time is spent. It reports; it does not stop the run. See the note below
# for why the confirmed rule is unsatisfiable for an extractive coref system.
STRATUM_PREFLIGHT_MIN = 30      # per dataset, best coref method - a reporting threshold

print("=" * 88)
print("STRATUM PREFLIGHT - built before any encoding")
print("=" * 88)
print(f"{'dataset':<10} {'coref':<10} {'queries':>8} {'confirmed':>10} {'+pronoun':>10}")
_pf_best = {}
for ds in DS_NAMES:
    for v in COREF_VARIANTS:
        n_conf = sum(1 for q in QIDS[ds] if CRIT[ds][v][q])
        n_pron = sum(1 for q in QIDS[ds] if CRIT[ds][v][q] and GOLD_HAS_PRON[ds][q])
        _pf_best[ds] = max(_pf_best.get(ds, 0), n_conf)
        print(f"{ds:<10} {v:<10} {len(QIDS[ds]):>8} {n_conf:>10} {n_pron:>10}")
print("-" * 88)
print(f"reporting threshold: {STRATUM_PREFLIGHT_MIN} confirmed queries per dataset under at")
print("least one coref method. 'confirmed' = the rewrite supplied a query content word that no")
print("gold passage had. '+pronoun' = of those, the ones whose gold passages contain a pronoun.")
print("The analysis reports ALL QUERIES (the paper's own condition) and the gold_has_pronoun")
print("subset (the coreference claim); confirmed is carried as a diagnostic.")
# WARNS, does not halt. The confirmed rule requires a gold passage to LACK a query
# content word AND the rewrite to ADD it. LingMess is extractive - every string it inserts
# is copied from the same passage - so the passage's content-word SET cannot grow and
# confirmed is 0 by construction, on any corpus. The LLM is generative, so it CAN add a
# word that was never in the passage; but a word it adds came from its parameters, not the
# document, so a non-zero LLM stratum is the set of queries where it hallucinated a
# query-relevant term into a gold passage. That is a contamination diagnostic, not evidence
# for coreference. Either way the run proceeds: the aggregate arm is the paper's own
# condition and does not depend on this stratum.
for ds in DS_NAMES:
    if _pf_best[ds] < STRATUM_PREFLIGHT_MIN:
        print(f"NOTE [{ds}]: confirmed stratum is {_pf_best[ds]} "
              f"({'empty' if _pf_best[ds] == 0 else 'under ' + str(STRATUM_PREFLIGHT_MIN)}). "
              f"Expected for lingmess - the rule is unsatisfiable for an extractive system. "
              f"The analysis reports all queries and the gold_has_pronoun subset; the "
              f"confirmed stratum is carried as a diagnostic only.")
print("=" * 88)
log.info("stage4: stratum preflight is advisory - best confirmed per dataset = %s",
         {d: _pf_best[d] for d in DS_NAMES})

# ---------------------------------------------------------------- OOM-tolerant encoding
def encode_retry(texts, tok, mdl, cfg, prefix, desc):
    """encode(), but a CUDA OOM halves the batch and retries instead of killing the run.
    Down to batch=1, then the OOM is re-raised. A successful downshift STICKS for the rest
    of this model, so the next variant does not walk into the same wall. The retry redoes
    the whole encode at the smaller batch - simpler than resuming mid-array, and an OOM is
    rare enough that the wasted work is cheaper than the complexity."""
    bs = int(cfg.get("batch", EMBED_BATCH))
    while True:
        cfg["batch"] = bs
        try:
            return encode(texts, tok, mdl, cfg, prefix, desc)
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if bs <= 1:
                print(f"OOM at batch=1 on {desc} - cannot go lower.")
                log.error("stage4: OOM at batch=1 on %s - re-raising", desc)
                raise
            nxt = max(1, bs // 2)
            print(f"OOM on {desc} at batch={bs} -> retrying at batch={nxt}")
            log.warning("stage4: OOM on %s at batch=%d - halving to %d and retrying",
                        desc, bs, nxt)
            bs = nxt

# ---------------------------------------------------------------- raw-row I/O
def prune_jsonl(keep_models):
    """Drop rows from a model that did not finish (and any torn final line), so a resumed
    session cannot double-count. Only fully completed models survive."""
    if not os.path.exists(PER_QUERY_JSONL):
        return 0, 0
    kept, dropped = [], 0
    with open(PER_QUERY_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                dropped += 1          # truncated last line from a kill mid-write
                continue
            if r.get("model") in keep_models:
                kept.append(line)
            else:
                dropped += 1
    if dropped:
        fd, tmp = tempfile.mkstemp(dir=DATA, suffix=".tmp")
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            for line in kept:
                f.write(line + "\n")
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, PER_QUERY_JSONL)
        log.warning("stage4: pruned %d row(s) from unfinished models; %d kept", dropped, len(kept))
    return len(kept), dropped

def append_rows(rows):
    """Append one model's rows and force them to disk. If the session dies after this
    returns, every model up to and including this one stays usable."""
    with open(PER_QUERY_JSONL, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())
    log.info("stage4: appended %d rows -> %s (%.1f MB)", len(rows), PER_QUERY_JSONL,
             os.path.getsize(PER_QUERY_JSONL) / 1e6)

# ---------------------------------------------------------------- sanity floor
FLOOR_FAILURES, floor_ok = [], {}

def sanity_floor(mname, cfg, ds, met):
    """QUARANTINING, not halting. Per dataset: Recall@10 >= 0.50 on scifact, nDCG@10 >= 0.25
    on nfcorpus, measured on the ORIGINAL (non-coref) corpus with dense retrieval. Below the
    floor is a misconfiguration - almost always a wrong or missing prompt prefix - and every
    coref delta measured on top of it would be noise around a broken baseline.

    This used to raise. In an unattended run raising here throws away every model that had
    not run yet, to punish one that was misconfigured. So the verdict is now recorded
    instead: the model's rows are still written and still carry floor_pass=False, the
    analysis excludes them from every headline table, and the failure is reported loudly in
    the final summary. Nothing below the floor is ever reported as a result - which is what
    the halt was actually protecting - but one bad encoder no longer costs the session.

    Returns (ok, value, metric, threshold)."""
    metric, thresh = SANITY_FLOOR[ds]
    val = float(met[metric].mean())
    log.info("stage4: FLOOR %s/%s original-dense %s=%.4f (floor %.2f) prefix_scheme=%s",
             mname, ds, metric, val, thresh, cfg["prefix_scheme"])
    if val < thresh:
        print("=" * 88)
        print("SANITY FLOOR FAILURE - THIS IS A MISCONFIGURATION, NOT A RESULT")
        print("=" * 88)
        print(f"  model          : {mname}   ({cfg['hf']})")
        print(f"  prefix scheme  : {cfg['prefix_scheme']}")
        print(f"  query prefix   : {cfg['query_prefix']!r}")
        print(f"  passage prefix : {cfg['passage_prefix']!r}")
        print(f"  append_eos     : {bool(cfg.get('append_eos'))}    pooling: {cfg['pooling']}"
              f"    dtype: {cfg['dtype']}")
        print(f"  dataset        : {ds}")
        print(f"  {metric:<14} : {val:.4f}   floor {thresh:.2f}   "
              f"(ORIGINAL corpus, dense retrieval)")
        if cfg.get("floor_hint"):
            print(f"  hint           : {cfg['floor_hint']}")
        print("=" * 88)
        print("  This model is QUARANTINED: its rows are still written and flagged")
        print("  floor_pass=false, the analysis excludes it from every headline table, and")
        print("  the run continues with the next model.")
        print("=" * 88)
        log.error("stage4: FLOOR FAILURE %s/%s %s=%.4f < %.2f - quarantined, run continues",
                  mname, ds, metric, val, thresh)
    _CHECKS.append({"label": f"SANITY FLOOR {mname}/{ds}: original-corpus {metric} >= "
                             f"{thresh:.2f}", "pass": bool(val >= thresh),
                    "detail": f"{metric}={val:.4f} prefix_scheme={cfg['prefix_scheme']} "
                              f"pooling={cfg['pooling']} dtype={cfg['dtype']}"})
    return bool(val >= thresh), val, metric, thresh

# ---------------------------------------------------------------- resume
_done = load_stage(DONE_MODELS_CK) or {"models": []}
done_models = [m for m in _done.get("models", []) if m in EMBED_MODELS]
n_kept, n_dropped = prune_jsonl(set(done_models))
log.info("stage4: resume | completed models=%s | rows on disk=%d", done_models, n_kept)

timings = {}
session_models, session_secs, session_weight = [], 0.0, 0.0
STAGE4_STOPPED_EARLY = False

for mname, cfg in EMBED_MODELS.items():
    if mname in done_models:
        log.info("stage4: %s already complete - skipping (rows are on disk)", mname)
        continue

    # Models run cheapest-first, so stopping here leaves the largest possible set of
    # completed models on disk. Every finished model is fully analysable on its own.
    if deadline_hit(DEADLINE_MARGIN_S):
        STAGE4_STOPPED_EARLY = True
        log.warning("stage4: session budget spent before %s - stopping with %d model(s) done",
                    mname, len(done_models))
        print("=" * 88)
        print("SESSION BUDGET REACHED IN STAGE 4 - NOT A FAILURE")
        print(f"  completed models : {done_models}")
        print(f"  not run          : {[m for m in EMBED_MODELS if m not in done_models]}")
        print("  Their rows are fsynced in per_query_results.jsonl and the analysis below")
        print("  runs on them. Attach this kernel's output and push again to add the rest.")
        print("=" * 88)
        break

    t_model = time.time()
    tok, mdl = load_encoder(mname, cfg)
    rows = []

    for ds in DS_NAMES:
        qv = encode_retry([QTXT[ds][q] for q in QIDS[ds]], tok, mdl, cfg,
                          cfg["query_prefix"], f"q[{mname}/{ds}]")
        assert_emb_sane(qv, f"{mname}/{ds}/queries")
        check(qv.shape[0] == len(QIDS[ds]), f"{mname}/{ds}: one vector per query", qv.shape)

        per_variant = {}
        for v in VARIANTS:
            t0 = time.time()
            if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
            pv = encode_retry(CORPUS_TEXT[ds][v], tok, mdl, cfg,
                              cfg["passage_prefix"], f"p[{mname}/{ds}/{v}]")
            assert_emb_sane(pv, f"{mname}/{ds}/{v}")
            check(pv.shape[0] == len(P[ds]), f"{mname}/{ds}/{v}: one vector per passage",
                  pv.shape)
            d_rl = dense_ranklist(qv, pv)
            h_rl = weighted_rrf(d_rl, BM25_RL[ds])
            per_variant[v] = {"dense": (d_rl, per_query_metrics(ds, d_rl)),
                              "hybrid": (h_rl, per_query_metrics(ds, h_rl))}
            if v == "original":
                # quarantining, before any coref variant of this model is embedded
                _fok, _fval, _fmet, _fthr = sanity_floor(mname, cfg, ds,
                                                         per_variant[v]["dense"][1])
                floor_ok[(mname, ds)] = _fok
                if not _fok:
                    FLOOR_FAILURES.append({"model": mname, "dataset": ds, "metric": _fmet,
                                           "value": round(float(_fval), 4), "threshold": _fthr,
                                           "prefix_scheme": cfg["prefix_scheme"],
                                           "pooling": cfg["pooling"], "dtype": cfg["dtype"],
                                           "hint": cfg.get("floor_hint", "")})
            dt, pk = time.time() - t0, peak_vram_gb()
            timings[f"{mname}|{ds}|{v}"] = {"wall_clock_s": round(dt, 1),
                                            "peak_vram_gb": round(pk, 2), "seed": SEED,
                                            "n_passages": len(P[ds]),
                                            "pooling": cfg["pooling"], "dtype": cfg["dtype"],
                                            "prefix_scheme": cfg["prefix_scheme"],
                                            "batch_used": int(cfg.get("batch", EMBED_BATCH))}
            log.info("stage4: %s/%s/%s encoded+scored in %.0fs | peak VRAM %.2f GB",
                     mname, ds, v, dt, pk)
            del pv

        # ---- raw per-query rows: one per (model, dataset, condition, retrieval, query) ----
        for v in VARIANTS:
            condition = "baseline" if v == "original" else "coref"
            method    = "none" if v == "original" else v
            for r in ("dense", "hybrid"):
                rl, met = per_variant[v][r]
                for i, q in enumerate(QIDS[ds]):
                    # A baseline row is shared by both coref methods, so its stratum flag is
                    # the union; the analysis always filters on the COREF row's flag.
                    in_str = (any(CRIT[ds][cv][q] for cv in COREF_VARIANTS)
                              if v == "original" else CRIT[ds][v][q])
                    rows.append({
                        "model": mname, "model_role": cfg["role"], "model_year": cfg["year"],
                        "pooling": cfg["pooling"], "prefix_scheme": cfg["prefix_scheme"],
                        "dataset": ds, "condition": condition, "coref_method": method,
                        "retrieval": r, "qid": q,
                        "top50": top_doc_ids(rl[i], ds),
                        "recall_10": round(float(met["recall_10"][i]), 6),
                        "ndcg_10": round(float(met["ndcg_cut_10"][i]), 6),
                        "mrr": round(float(met["recip_rank"][i]), 6),
                        "floor_pass": bool(floor_ok.get((mname, ds), True)),
                        "in_confirmed_stratum": bool(in_str),
                        "gold_has_pronoun": bool(GOLD_HAS_PRON[ds][q]),
                        "n_gold": int(N_GOLD[ds][q])})
        del per_variant, qv

    del mdl, tok
    free_vram(f"stage4/{mname}")

    # ---- durable: rows first (fsynced), then the completion marker ----
    append_rows(rows)
    done_models.append(mname)
    save_stage(DONE_MODELS_CK, {"models": done_models, "seed": SEED, "env": ENV,
                                "n_rows_last_append": len(rows)})
    el_model = time.time() - t_model
    session_models.append(mname); session_secs += el_model
    session_weight += cfg["cost_weight"]
    log.info("stage4: %s COMPLETE in %.0fs (%.2f h) | %d models done", mname, el_model,
             el_model / 3600, len(done_models))
    del rows

    # ---- item 11: after the first model of this session, project the whole run ----
    if len(session_models) == 1:
        per_w = el_model / max(cfg["cost_weight"], 1e-9)
        w_all = sum(c["cost_weight"] for c in EMBED_MODELS.values())
        w_left = sum(c["cost_weight"] for m, c in EMBED_MODELS.items()
                     if m not in done_models)
        print("=" * 88)
        print("WALL-CLOCK PROJECTION (after model 1 of this session)")
        print("=" * 88)
        print(f"  {mname} took {el_model / 60:.1f} min (cost_weight {cfg['cost_weight']}).")
        print(f"  {'model':<24} {'cost_weight':>11} {'projected':>12}  status")
        for m, c in EMBED_MODELS.items():
            proj = per_w * c["cost_weight"] / 60
            print(f"  {m:<24} {c['cost_weight']:>11.1f} {proj:>10.1f} m  "
                  f"{'done' if m in done_models else 'pending'}")
        print(f"  WHOLE ROSTER     : ~{per_w * w_all / 3600:.1f} h")
        print(f"  REMAINING NOW    : ~{per_w * w_left / 3600:.1f} h")
        print("  Kaggle cuts a session at 12h. If REMAINING exceeds what is left of this")
        print("  session, stop after any model: its rows are already fsynced, and a rerun")
        print("  resumes at the next model.")
        print("  This is a projection from declared cost_weights, not a measurement.")
        print("=" * 88)

# ---------------------------------------------------------------- run provenance
# NOT a check(). A partial roster is a legitimate outcome of a budgeted unattended run, and
# raising here would discard the analysis of models that DID finish, at the very end, after
# all their GPU time was already spent. The completeness of the roster is recorded instead
# and reported in the final summary; the analysis reads whatever rows exist.
STAGE4_MODELS_MISSING = [m for m in EMBED_MODELS if m not in done_models]
if STAGE4_MODELS_MISSING:
    save_run_state("stage4", {"done_models": done_models,
                              "missing_models": STAGE4_MODELS_MISSING,
                              "stopped_early": STAGE4_STOPPED_EARLY})
    log.warning("stage4: INCOMPLETE roster - missing %s", STAGE4_MODELS_MISSING)
else:
    log.info("stage4: complete roster - %d models", len(done_models))
if FLOOR_FAILURES:
    log.error("stage4: %d sanity-floor failure(s) quarantined: %s", len(FLOOR_FAILURES),
              [(f["model"], f["dataset"]) for f in FLOOR_FAILURES])
save_stage("stage4_run_meta.json", {
    "stage": "stage4_run_meta", "datasets": DS_NAMES, "seed": SEED,
    "per_query_file": PER_QUERY_JSONL,
    "n_queries": {ds: len(QIDS[ds]) for ds in DS_NAMES},
    "n_passages": {ds: len(P[ds]) for ds in DS_NAMES},
    "embed_models": {k: {"hf": v["hf"], "pooling": v["pooling"], "dtype": v["dtype"],
                         "role": v["role"], "year": v["year"], "params_m": v["params_m"],
                         "prefix_scheme": v["prefix_scheme"],
                         "query_prefix": v["query_prefix"],
                         "passage_prefix": v["passage_prefix"],
                         "append_eos": bool(v.get("append_eos"))}
                     for k, v in EMBED_MODELS.items()},
    "run_order": list(EMBED_MODELS),
    "done_models": done_models,
    "missing_models": STAGE4_MODELS_MISSING,
    "stopped_early": bool(STAGE4_STOPPED_EARLY),
    "floor_failures": FLOOR_FAILURES,
    "stage0_gate": STAGE0_GATE,
    "stratum_gate": STRATUM_GATE,
    "llm_coref_model": LLM_MODEL,
    "llm_paper_model": LLM_PAPER_MODEL,
    "llm_is_paper_model": LLM_MODEL == LLM_PAPER_MODEL,
    "anaphor_skip_active": bool(SKIP_ACTIVE),
    "batch_mode": bool(BATCH),
    "session_budget_h": SESSION_BUDGET_H,
    "elapsed_h": round((time.time() - RUN_STARTED_AT) / 3600, 2),
    "sanity_floor": {ds: {"metric": SANITY_FLOOR[ds][0], "threshold": SANITY_FLOOR[ds][1]}
                     for ds in DS_NAMES},
    "coref_methods": {"lingmess": S2[DS_NAMES[0]]["method"], "llm": S3[DS_NAMES[0]]["method"]},
    "prompt_fingerprint": PROMPT_FINGERPRINT,
    "retrieval": {"rrf_k": RRF_K, "dense_weight": DENSE_WEIGHT, "run_depth": RUN_DEPTH,
                  "saved_top_n": SAVE_TOPN},
    "strata_counts": {ds: {"eligible": S1[ds]["strata"]["n_coref_eligible"],
                           "eligible_no_exclusion":
                               S1[ds]["strata"]["n_coref_eligible_no_exclusion"],
                           **{v: NCRIT[ds][v] for v in COREF_VARIANTS},
                           "gold_has_pronoun": int(sum(GOLD_HAS_PRON[ds].values()))}
                      for ds in DS_NAMES},
    "strata_evidence": {ds: {v: EV[ds][v] for v in COREF_VARIANTS} for ds in DS_NAMES},
    "coref_stats": {ds: {"lingmess": S2[ds]["stats"],
                         "llm": {k: v for k, v in S3[ds]["stats"].items()
                                 if k != "failure_detail"}} for ds in DS_NAMES},
    "corpus_pronoun_share": {ds: S1[ds]["strata"]["corpus_pronoun_share"] for ds in DS_NAMES},
    "timings": timings, "env": ENV, "checks": checks_summary()})

log.info("stage4 complete | models=%s | raw rows -> %s", done_models, PER_QUERY_JSONL)

## Analysis — CPU only, reads `per_query_results.jsonl` and nothing else

No GPU, no model, no dataset download. It can be run on a fresh CPU session, or re-run after a
partial GPU run: whatever models finished are in the file, and the analysis reports exactly those.

**Per dataset. Nothing is pooled across `scifact` and `nfcorpus`** — they differ in gold density
(~1.1 vs ~38 gold per query) and in relevance grading, so a pooled average would be a weighted
average of two different measurement problems.

### What is computed

1. **Delta against baseline score.** Per query, `delta = coref − baseline` on nDCG@10, regressed
   against that query's **baseline** score, per model per dataset per coref method. Reported as OLS
   slope and Spearman rho, each with a **bootstrap 95% CI over 10,000 query resamples** (seed 42),
   and plotted. A strongly negative slope means the gain is regression to the mean — coref helping
   exactly where the baseline was weakest — rather than a uniform improvement.
2. **Paired Wilcoxon signed-rank** on the per-query deltas, with the **matched-pairs rank-biserial
   correlation** as the effect size and the median delta beside it. The p-value alone is not
   reported; a p without an effect size cannot distinguish a tiny real shift from a useful one.

### Every result is reported twice

* **confirmed stratum** — queries where the rewrite actually supplied a query content word that no
  gold passage had. This is the **upper bound**: the largest set where coref could plausibly matter.
* **confirmed ∩ `gold_has_pronoun`** — the subset whose gold passages actually contain a pronoun.
  **This is the coreference claim.** A gain that appears in the first but not the second is not
  coming from coreference resolution.

Both are reported even when small, with `n` in every row. Where `n` is too small to fit a line
(< 8 queries), the regression is reported as underpowered rather than fitted — the emptiness is
itself a result, and the raw rows remain in the file for anyone who wants a different cut.

In [ ]:
# ===== ANALYSIS - CPU ONLY. Reads per_query_results.jsonl and nothing else. No GPU, no =====
# ===== model, no dataset download. Safe to run on a fresh CPU session or after a       =====
# ===== partial GPU run: whatever models finished are in the file.                       =====
import os, json, math
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

pd.set_option("display.width", 230); pd.set_option("display.max_rows", 600)

JSONL   = globals().get("PER_QUERY_JSONL", "/kaggle/working/test-8-data/per_query_results.jsonl")
OUT_DIR = os.path.dirname(JSONL) or "."
BOOT_N  = int(globals().get("BOOTSTRAP_N", 10000))
SEED_A  = int(globals().get("SEED", 42))
METRICS = ["ndcg_10", "recall_10", "mrr"]
PRIMARY = "ndcg_10"       # delta-vs-baseline regression and Wilcoxon run on this metric
RETR    = "dense"         # primary analysis is dense; hybrid appears in the absolute table
MIN_FIT = 8               # below this, a slope/rho is reported as underpowered, not fitted
# Scope 1 is the paper's own condition and is benchmark-comparable; scope 2 is the
# coreference claim, defined from ORIGINAL text before any retrieval, so it is neither
# post-hoc nor outcome-dependent. Scope 3 is a diagnostic: for lingmess it is empty by
# construction, and for the llm it is the set of queries where a term absent from the gold
# passage was introduced by the rewrite - i.e. hallucinated in. Read it as contamination,
# never as evidence for coreference.
SCOPES  = [("all queries", "all"),
           ("gold_has_pronoun", "pronoun"),
           ("confirmed stratum (diagnostic)", "confirmed")]

# ---------------------------------------------------------------- load raw rows
recs, bad_lines = [], 0
with open(JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            recs.append(json.loads(line))
        except json.JSONDecodeError:
            bad_lines += 1
assert recs, f"no rows in {JSONL}"
df = pd.DataFrame(recs)
if "top50" in df.columns:
    df = df.drop(columns=["top50"])       # doc ids stay in the file; the frame stays small
del recs

# ---------------------------------------------------------------- sanity-floor quarantine
# Stage 4 no longer halts on a model that falls below its floor: it writes that model's rows
# flagged floor_pass=false and moves on, so one misconfigured encoder cannot cost a whole
# unattended session. The exclusion happens HERE instead. Nothing below the floor reaches a
# table, which is what the halt was really protecting - but the models are NAMED, never
# silently dropped.
if "floor_pass" in df.columns:
    _fp = df["floor_pass"].fillna(True).astype(bool)
    QUARANTINED = sorted({(r.model, r.dataset)
                          for r in df.loc[~_fp, ["model", "dataset"]].itertuples()})
    if QUARANTINED:
        print("=" * 100)
        print("QUARANTINED BY THE SANITY FLOOR - excluded from every table below")
        for m, d in QUARANTINED:
            print(f"  {m} / {d}: below its floor on the ORIGINAL corpus. A coref delta measured")
            print("     on top of a broken baseline is noise, not a result.")
        print("=" * 100)
    df = df[_fp].copy()
    assert len(df), "every row is below its sanity floor - nothing is analysable"
else:
    QUARANTINED = []

MODELS   = list(dict.fromkeys(df["model"]))
DATASETS = list(dict.fromkeys(df["dataset"]))
METHODS  = [m for m in ["lingmess", "llm"] if m in set(df["coref_method"])]
MODEL_META = (df[["model", "model_role", "model_year", "pooling", "prefix_scheme"]]
              .drop_duplicates("model").set_index("model"))

print("=" * 100)
print(f"ANALYSIS SOURCE: {JSONL}  ({len(df):,} rows"
      + (f", {bad_lines} unparsable line(s) skipped" if bad_lines else "") + ")")
print("=" * 100)
print(f"{'model':<24} {'role':<17} {'year':>5} {'pooling':>8} {'prefix scheme':<24} datasets")
for m in MODELS:
    meta = MODEL_META.loc[m]
    dss = sorted(set(df.loc[df['model'] == m, 'dataset']))
    print(f"{m:<24} {meta['model_role']:<17} {int(meta['model_year']):>5} "
          f"{meta['pooling']:>8} {meta['prefix_scheme']:<24} {dss}")
print(f"\ncoref methods present: {METHODS}   retrieval present: "
      f"{sorted(set(df['retrieval']))}")
print("Per dataset only. Nothing is pooled across scifact and nfcorpus.")

for ds in DATASETS:
    sub = df[(df["dataset"] == ds) & (df["condition"] == "coref") &
             (df["retrieval"] == RETR) & (df["model"] == MODELS[0])]
    for meth in METHODS:
        s = sub[sub["coref_method"] == meth]
        if not len(s):
            continue
        conf = s["in_confirmed_stratum"].astype(bool)
        pron = s["gold_has_pronoun"].astype(bool)
        print(f"  [{ds}/{meth}] queries={len(s)}  confirmed={int(conf.sum())}  "
              f"confirmed+pronoun={int((conf & pron).sum())}  "
              f"gold_has_pronoun(all)={int(pron.sum())}  "
              f"mean n_gold={s['n_gold'].mean():.1f}")

# ---------------------------------------------------------------- pairing
def paired(model, ds, method, retrieval=RETR):
    """Baseline and coref rows for the same query, side by side, with the per-query delta."""
    sel = (df["model"] == model) & (df["dataset"] == ds) & (df["retrieval"] == retrieval)
    b = df[sel & (df["condition"] == "baseline")][["qid"] + METRICS]
    c = df[sel & (df["coref_method"] == method)][
        ["qid"] + METRICS + ["in_confirmed_stratum", "gold_has_pronoun", "n_gold"]]
    if not len(b) or not len(c):
        return pd.DataFrame()
    m = b.merge(c, on="qid", suffixes=("_base", "_coref"), validate="one_to_one")
    for k in METRICS:
        m["d_" + k] = m[k + "_coref"] - m[k + "_base"]
    return m

def scoped(m, skey):
    if not len(m) or skey == "all":
        return m
    if skey == "pronoun":
        return m[m["gold_has_pronoun"].astype(bool)]
    return m[m["in_confirmed_stratum"].astype(bool)]

# ---------------------------------------------------------------- statistics
def ols_slope(x, y):
    vx = float(np.var(x))
    if vx < 1e-12:
        return float("nan")
    return float(np.cov(x, y, bias=True)[0, 1] / vx)

def spearman_rho(x, y):
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    if np.std(rx) < 1e-12 or np.std(ry) < 1e-12:
        return float("nan")
    return float(np.corrcoef(rx, ry)[0, 1])

def boot_ci(fn, *arrays, n=BOOT_N, seed=SEED_A):
    """Percentile bootstrap over QUERIES, resampled with replacement. Fixed seed, and a
    fresh generator per call so a result does not depend on evaluation order."""
    k = len(arrays[0])
    if k < MIN_FIT:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, k, size=(n, k))
    vals = np.array([fn(*[a[i] for a in arrays]) for i in idx], dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size < 100:
        return (float("nan"), float("nan"))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return (float(lo), float(hi))

def wilcoxon_paired(d):
    """Paired Wilcoxon signed-rank on the per-query deltas, with the matched-pairs
    rank-biserial correlation as the effect size. p alone cannot tell a tiny real shift
    from a useful one, so it is never reported on its own."""
    d = np.asarray(d, dtype=float)
    nz = d[d != 0.0]
    out = {"n": int(d.size), "n_nonzero": int(nz.size),
           "median_delta": float(np.median(d)) if d.size else float("nan"),
           "mean_delta": float(d.mean()) if d.size else float("nan"),
           "rank_biserial": float("nan"), "p": float("nan"), "W": float("nan")}
    if nz.size == 0:
        out["rank_biserial"] = 0.0
        return out
    ranks = stats.rankdata(np.abs(nz))
    rp, rm = float(ranks[nz > 0].sum()), float(ranks[nz < 0].sum())
    out["rank_biserial"] = (rp - rm) / (rp + rm) if (rp + rm) > 0 else 0.0
    try:
        W, p = stats.wilcoxon(nz, zero_method="wilcox", alternative="two-sided")
        out["W"], out["p"] = float(W), float(p)
    except Exception as e:                       # all-zero or degenerate input
        print(f"    (wilcoxon skipped: {e})")
    return out

def fmt_ci(lo, hi):
    return "n/a" if not (np.isfinite(lo) and np.isfinite(hi)) else f"[{lo:+.4f}, {hi:+.4f}]"

# ---------------------------------------------------------------- absolute scores
arows = []
for m in MODELS:
    for ds in DATASETS:
        for retr in sorted(set(df["retrieval"])):
            for meth in ["none"] + METHODS:
                s = df[(df["model"] == m) & (df["dataset"] == ds) &
                       (df["retrieval"] == retr) & (df["coref_method"] == meth)]
                if not len(s):
                    continue
                arows.append({"model": m, "pooling": MODEL_META.loc[m, "pooling"],
                              "dataset": ds, "retrieval": retr,
                              "condition": "baseline" if meth == "none" else f"coref:{meth}",
                              "N": len(s),
                              **{k: round(float(s[k].mean()), 4) for k in METRICS}})
abs_df = pd.DataFrame(arows)
print("\n" + "=" * 100)
print("ABSOLUTE SCORES (per dataset; no pooling)")
print("=" * 100)
print(abs_df.to_string(index=False))

# ---------------------------------------------------------------- 1) delta vs baseline
reg_rows = []
for m in MODELS:
    for ds in DATASETS:
        for meth in METHODS:
            pm = paired(m, ds, meth)
            if not len(pm):
                continue
            for slabel, skey in SCOPES:
                s = scoped(pm, skey)
                x = s[PRIMARY + "_base"].to_numpy(dtype=float)
                y = s["d_" + PRIMARY].to_numpy(dtype=float)
                row = {"model": m, "dataset": ds, "coref": meth, "scope": slabel, "N": len(s)}
                if len(s) < MIN_FIT:
                    row.update({"slope": None, "slope 95% CI": "underpowered",
                                "spearman rho": None, "rho 95% CI": "underpowered",
                                "mean baseline": round(float(x.mean()), 4) if len(s) else None})
                else:
                    sl = ols_slope(x, y)
                    rho = spearman_rho(x, y)
                    sl_ci = boot_ci(ols_slope, x, y)
                    rho_ci = boot_ci(spearman_rho, x, y)
                    row.update({"slope": round(sl, 4), "slope 95% CI": fmt_ci(*sl_ci),
                                "spearman rho": round(rho, 4), "rho 95% CI": fmt_ci(*rho_ci),
                                "mean baseline": round(float(x.mean()), 4)})
                reg_rows.append(row)
reg_df = pd.DataFrame(reg_rows)
print("\n" + "=" * 100)
print(f"1) DELTA vs BASELINE - OLS slope and Spearman rho of (coref - baseline) {PRIMARY}")
print(f"   against the query's baseline {PRIMARY}. Bootstrap 95% CI, {BOOT_N:,} query "
      f"resamples, seed {SEED_A}. Dense retrieval.")
print("=" * 100)
print(reg_df.to_string(index=False) if len(reg_df) else "  (no paired rows)")
print("A slope near -1 means the coref 'gain' is regression to the mean: it appears where the")
print("baseline was weakest and vanishes where the baseline was already right. A slope near 0")
print("with a CI excluding it is a uniform shift. Read the CI, not the point estimate.")

# ---------------------------------------------------------------- 2) paired Wilcoxon
wil_rows = []
for m in MODELS:
    for ds in DATASETS:
        for meth in METHODS:
            pm = paired(m, ds, meth)
            if not len(pm):
                continue
            for slabel, skey in SCOPES:
                s = scoped(pm, skey)
                d = s["d_" + PRIMARY].to_numpy(dtype=float)
                w = wilcoxon_paired(d)
                lo, hi = boot_ci(lambda a: float(np.mean(a)), d) if len(d) else (np.nan, np.nan)
                wil_rows.append({
                    "model": m, "dataset": ds, "coref": meth, "scope": slabel,
                    "N": w["n"], "n!=0": w["n_nonzero"],
                    "mean delta": round(w["mean_delta"], 4) if np.isfinite(w["mean_delta"]) else None,
                    "mean 95% CI": fmt_ci(lo, hi),
                    "median delta": round(w["median_delta"], 4) if np.isfinite(w["median_delta"]) else None,
                    "rank-biserial": round(w["rank_biserial"], 4) if np.isfinite(w["rank_biserial"]) else None,
                    "p": (f"{w['p']:.4g}" if np.isfinite(w["p"]) else "n/a")})
wil_df = pd.DataFrame(wil_rows)
print("\n" + "=" * 100)
print(f"2) PAIRED WILCOXON SIGNED-RANK on per-query delta ({PRIMARY}, dense), confirmed")
print("   stratum only. Effect size = matched-pairs rank-biserial correlation; the median")
print(f"   delta and a bootstrap mean CI ({BOOT_N:,} resamples) sit beside it. p is never alone.")
print("=" * 100)
print(wil_df.to_string(index=False) if len(wil_df) else "  (no paired rows)")
print("rank-biserial: +1 every query improved, -1 every query degraded, 0 balanced.")
print("A significant p with a rank-biserial near zero is a shift too small to act on.")

# ---------------------------------------------------------------- the two scopes, side by side
print("\n" + "=" * 100)
print("3) THE PRONOUN SUBSET IS THE COREFERENCE CLAIM")
print("=" * 100)
print("The confirmed stratum is the upper bound - every query where the rewrite supplied a")
print("missing query term. The pronoun subset is the part of it where a pronoun actually")
print("existed to resolve. A gain present in the first and absent in the second did not come")
print("from coreference resolution.")
if len(wil_df):
    try:
        piv = wil_df.pivot_table(index=["model", "dataset", "coref"], columns="scope",
                                 values=["N", "mean delta", "rank-biserial"], aggfunc="first")
        print(piv.to_string())
    except Exception as e:                      # never let a display quirk kill the analysis
        print(f"  (pivot unavailable: {e}; the same numbers are in the Wilcoxon table above)")

# ---------------------------------------------------------------- ladder / pooling views
print("\n" + "=" * 100)
print("4) MEAN-POOLING LADDER (generation varies, pooling constant) and POOLING CONTRAST")
print("=" * 100)
if len(wil_df):
    view = wil_df.merge(MODEL_META.reset_index(), on="model", how="left")
    lad = view[view["model_role"] == "ladder"].sort_values(
        ["dataset", "coref", "scope", "model_year"])
    _n_rungs = lad["model"].nunique() if len(lad) else 0
    print("LADDER (mean pooling throughout; the registered hypothesis is that NEWER models")
    print("gain LESS. It is being tested against, not assumed - Jang et al. Table 1 mildly")
    print("contradicts it.)")
    if _n_rungs < 3:
        print(f"UNDERPOWERED: only {_n_rungs} rung(s) present. google/embeddinggemma-300m, the")
        print("2025 rung, was dropped for GPU cost, so this ladder spans 2021 -> 2022 and both")
        print("models are tiny. It CANNOT answer the generation hypothesis. The rows below are")
        print("reported for completeness; the hypothesis stays REGISTERED BUT NOT ANSWERED.")
    print(lad[["dataset", "coref", "scope", "model", "model_year", "N",
               "mean delta", "mean 95% CI", "rank-biserial"]].to_string(index=False))
    pc = view.sort_values(["dataset", "coref", "scope", "pooling", "model"])
    print("\nPOOLING CONTRAST (the paper claims mean pooling gains more than CLS/last-token):")
    print(pc[["dataset", "coref", "scope", "pooling", "model", "N",
              "mean delta", "mean 95% CI", "rank-biserial"]].to_string(index=False))

# ---------------------------------------------------------------- power note
print("\n" + "=" * 100)
print("5) POWER NOTE (two-sided, alpha=0.05, power=0.80, paired) - from the per-query deltas")
print("=" * 100)
prows = []
for m in MODELS:
    for ds in DATASETS:
        for meth in METHODS:
            pm = paired(m, ds, meth)
            if not len(pm):
                continue
            for slabel, skey in SCOPES:
                d = scoped(pm, skey)["d_" + PRIMARY].to_numpy(dtype=float)
                if d.size < 2:
                    prows.append({"model": m, "dataset": ds, "coref": meth, "scope": slabel,
                                  "N": int(d.size), "observed delta": None, "sd": None,
                                  "N for 80% power": "n/a (empty)"})
                    continue
                sd = float(d.std(ddof=1)); dm = abs(float(d.mean()))
                need = 7.849 * (sd / dm) ** 2 if dm > 1e-9 else float("inf")
                prows.append({"model": m, "dataset": ds, "coref": meth, "scope": slabel,
                              "N": int(d.size), "observed delta": round(float(d.mean()), 4),
                              "sd": round(sd, 4),
                              "N for 80% power": ("n/a (delta~0)" if not math.isfinite(need)
                                                  else int(math.ceil(need)))})
pow_df = pd.DataFrame(prows)
print(pow_df.to_string(index=False) if len(pow_df) else "  (nothing to report)")

# ---------------------------------------------------------------- plots
png_files = []
for ds in DATASETS:
    for slabel, skey in SCOPES:
        nrow, ncol = len(MODELS), max(len(METHODS), 1)
        fig, axes = plt.subplots(nrow, ncol, figsize=(5.2 * ncol, 3.3 * nrow), squeeze=False)
        for i, m in enumerate(MODELS):
            for j, meth in enumerate(METHODS):
                ax = axes[i][j]
                s = scoped(paired(m, ds, meth), skey)
                ax.axhline(0.0, lw=0.8, color="0.6")
                if len(s):
                    x = s[PRIMARY + "_base"].to_numpy(dtype=float)
                    y = s["d_" + PRIMARY].to_numpy(dtype=float)
                    ax.scatter(x, y, s=16, alpha=0.65)
                    if len(s) >= MIN_FIT and np.var(x) > 1e-12:
                        sl = ols_slope(x, y)
                        icpt = float(y.mean() - sl * x.mean())
                        xs = np.linspace(float(x.min()), float(x.max()), 50)
                        ax.plot(xs, sl * xs + icpt, lw=1.6, color="crimson")
                        ax.set_title(f"{m} / {meth}\nn={len(s)}  slope={sl:+.3f}  "
                                     f"rho={spearman_rho(x, y):+.3f}", fontsize=9)
                    else:
                        ax.set_title(f"{m} / {meth}\nn={len(s)} (underpowered)", fontsize=9)
                else:
                    ax.set_title(f"{m} / {meth}\nn=0", fontsize=9)
                ax.set_xlabel(f"baseline {PRIMARY}", fontsize=8)
                ax.set_ylabel(f"delta {PRIMARY}", fontsize=8)
                ax.tick_params(labelsize=7)
        fig.suptitle(f"{ds} - per-query delta vs baseline ({PRIMARY}, dense) - {slabel}",
                     fontsize=12)
        fig.tight_layout(rect=(0, 0, 1, 0.97))
        png = os.path.join(OUT_DIR, f"delta_vs_baseline__{ds}__{skey}.png")
        fig.savefig(png, dpi=120)
        png_files.append(png)
        plt.show()
        plt.close(fig)
print(f"\nplots written: {png_files}")

# ---------------------------------------------------------------- findings markdown
def _tbl(d):
    if not len(d):
        return "_(empty)_"
    cols = list(d.columns)
    out = ["| " + " | ".join(str(c) for c in cols) + " |",
           "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in d.iterrows():
        out.append("| " + " | ".join("" if pd.isna(r[c]) else str(r[c]) for c in cols) + " |")
    return "\n".join(out)

md = f"""## Measured Results (auto-generated from `per_query_results.jsonl`)

Source: `{JSONL}` ({len(df):,} raw per-query rows).
Primary metric: **{PRIMARY}**, dense retrieval. Bootstrap: **{BOOT_N:,}** query resamples,
seed **{SEED_A}**. **Per dataset only — nothing is pooled across scifact and nfcorpus.**

Models present: {", ".join(MODELS)}
Datasets present: {", ".join(DATASETS)} | coref methods: {", ".join(METHODS)}
{("Quarantined by the sanity floor (excluded above): "
  + ", ".join(f"{m}/{d}" for m, d in QUARANTINED)) if QUARANTINED else
 "No model fell below its sanity floor."}

Every result appears twice: the **confirmed stratum** (upper bound) and the
**confirmed + `gold_has_pronoun`** subset (the coreference claim).

### Absolute scores

{_tbl(abs_df)}

### Delta vs baseline (OLS slope, Spearman rho, bootstrap 95% CI)

{_tbl(reg_df)}

A slope near -1 is regression to the mean rather than a uniform gain.

### Paired Wilcoxon on per-query delta, with effect size

{_tbl(wil_df)}

`rank-biserial`: +1 = every query improved, -1 = every query degraded, 0 = balanced.

### Power

{_tbl(pow_df)}

### Plots

{chr(10).join("- `" + os.path.basename(p) + "`" for p in png_files)}
"""
with open(os.path.join(OUT_DIR, "test-8-results.md"), "w", encoding="utf-8") as f:
    f.write(md)
print("\nwrote", os.path.join(OUT_DIR, "test-8-results.md"))
print("ARTIFACTS:", sorted(os.listdir(OUT_DIR)))

## Run summary · the one file to read after an unattended push

Everything above is the run talking to itself. This cell writes `RESULTS.md` and
`run_summary.json`, which are what a person should open first after a batch job finishes: what ran,
what every gate decided, what was skipped or quarantined, what is missing, and how to resume if the
session budget cut it short.


In [ ]:
# ===== RUN SUMMARY - CPU only, defensive: works even if an earlier stage never ran =====
_G = globals()
def _g(k, d=None):
    v = _G.get(k, d)
    return d if v is None else v

_out   = _g("OUT_DIR", _g("DATA", "/kaggle/working/test-8-data"))
_done  = _g("done_models", [])
_miss  = _g("STAGE4_MODELS_MISSING", [])
_floor = _g("FLOOR_FAILURES", [])
_s0g   = _g("STAGE0_GATE", {})
_stg   = _g("STRATUM_GATE", {})
_S0    = _g("S0", {})
_S2    = _g("S2", {})
_S3    = _g("S3", {})
_dsn   = _g("DS_NAMES", [])
_elap  = round((time.time() - _g("RUN_STARTED_AT", time.time())) / 3600, 2)
_incomplete = bool(_miss) or bool(_g("STAGE4_STOPPED_EARLY", False))

summary = {
    "run": {"batch_mode": bool(_g("BATCH", False)), "elapsed_h": _elap,
            "session_budget_h": _g("SESSION_BUDGET_H", 0.0),
            "complete": not _incomplete, "seed": _g("SEED", 42),
            "env": _g("ENV", {})},
    "coref_llm": {"model": _g("LLM_MODEL", ""), "paper_model": _g("LLM_PAPER_MODEL", ""),
                  "is_paper_model": _g("LLM_MODEL", "") == _g("LLM_PAPER_MODEL", ""),
                  "quantised_4bit": bool(_g("LLM_LOAD_4BIT", False)),
                  "anaphor_skip_active": bool(_g("SKIP_ACTIVE", False))},
    "gates": {"stage0": _s0g, "stratum": _stg},
    "stage4": {"completed_models": _done, "missing_models": _miss,
               "floor_failures": _floor,
               "quarantined": [list(x) for x in _g("QUARANTINED", [])]},
    "coref_stats": {ds: {"lingmess": _S2.get(ds, {}).get("stats", {}),
                         "llm": {k: v for k, v in _S3.get(ds, {}).get("stats", {}).items()
                                 if k != "failure_detail"}} for ds in _dsn},
    "checks": _g("checks_summary", lambda: {})(),
}
save_stage("run_summary.json", summary)

_lines = []
_A = _lines.append
_A("# Test 8 - run summary")
_A("")
_A(f"- mode: **{'BATCH (unattended)' if summary['run']['batch_mode'] else 'interactive'}**")
_A(f"- elapsed: **{_elap} h** of a {summary['run']['session_budget_h']} h budget")
_A(f"- status: **{'COMPLETE' if not _incomplete else 'INCOMPLETE - see resume below'}**")
_A("")
_A("## What produced the coreference")
_A("")
_A(f"- LLM: `{summary['coref_llm']['model']}` "
   f"({'4-bit nf4' if summary['coref_llm']['quantised_4bit'] else 'fp16'})")
if not summary["coref_llm"]["is_paper_model"]:
    _A(f"- **Departure from the paper.** Jang et al. Appendix B use "
       f"`{summary['coref_llm']['paper_model']}`. This run used a smaller model of the same "
       f"family with the same Table 5 prompt, to fit a free-tier session. Stage 0 measured its "
       f"coref behaviour before Stage 3 spent anything; those numbers are below. This is a "
       f"replication of the METHOD, not of their exact configuration.")
if summary["coref_llm"]["anaphor_skip_active"]:
    _A("- **Anaphor skip active.** Passages containing no pronoun were kept verbatim without an "
       "LLM call. Stage 0 verified this model changed no pronoun-free reference chunk "
       f"({_S0.get('pron_free_changed', 0)} of {_S0.get('pron_free_total', 0)}).")
elif _g("SKIP_NO_ANAPHOR", False):
    _A(f"- Anaphor skip requested but **NOT active**: "
       f"{_S0.get('pron_free_changed', 0)} of {_S0.get('pron_free_total', 0)} pronoun-free "
       f"reference chunks were changed by this model, so every passage went to the LLM.")
_A("")
_A("## Gates")
_A("")
if _s0g:
    _A(f"- Stage 0 ({_s0g.get('mode', '?')}): changed {_s0g.get('changed')}, "
       f"failure rate {_s0g.get('fail_rate')}, hallucination rate {_s0g.get('halluc_rate')} "
       f"-> {'PASS' if not _s0g.get('failures') else 'FAIL: ' + '; '.join(_s0g['failures'])}")
if _stg:
    _A(f"- Stratum: {_stg.get('pooled_confirmed')} confirmed vs threshold "
       f"{_stg.get('min_required')} -> "
       f"{'UNDERPOWERED (recorded as a finding, run continued)' if _stg.get('underpowered') else 'ok'}")
_A("")
_A("## Stage 4 roster")
_A("")
_A(f"- completed: {', '.join(_done) if _done else 'none'}")
if _miss:
    _A(f"- **not run: {', '.join(_miss)}**")
if _floor:
    for f in _floor:
        _A(f"- **sanity-floor failure (quarantined)**: {f['model']} / {f['dataset']} "
           f"{f['metric']}={f['value']} < {f['threshold']} "
           f"(prefix_scheme={f['prefix_scheme']}, pooling={f['pooling']}, dtype={f['dtype']})")
else:
    _A("- no sanity-floor failures")
_A("")
_A("## Coreference stats per dataset")
_A("")
for ds in _dsn:
    _st = summary["coref_stats"].get(ds, {})
    _lm, _ll = _st.get("lingmess", {}), _st.get("llm", {})
    # 'changed' / 'pronoun_reduction' are the keys coref_stats actually writes. The pronoun
    # reduction is the one that says whether a coref arm DID coreference at all: a method can
    # rewrite half the corpus and still resolve nothing.
    _A(f"- `{ds}`: lingmess changed {_lm.get('changed', '?')} "
       f"(pronouns -{100 * _lm.get('pronoun_reduction', 0):.1f}%), "
       f"llm changed {_ll.get('changed', '?')} "
       f"(pronouns -{100 * _ll.get('pronoun_reduction', 0):.1f}%)"
       + (f", llm skipped (no anaphor) {_st.get('llm', {}).get('llm_skipped_no_anaphor')}"
          if _st.get("llm", {}).get("llm_skip_rule_active") else ""))
_A("")
_A("## Known limitations of THIS run")
_A("")
_A("- `google/embeddinggemma-300m` was dropped for GPU cost. The mean-pooling ladder therefore "
   "spans 2021 -> 2022 only, and the registered hypothesis that newer encoders gain less from "
   "coreference is **registered but not answered**. The paper's own pooling claim "
   "(mean vs CLS vs last-token) is unaffected and fully tested.")
if not summary["coref_llm"]["is_paper_model"]:
    _A("- The coreference model is smaller than the paper's. Absolute coref quality is therefore "
       "not comparable to theirs; the within-notebook A/B (none vs LingMess vs LLM) is unaffected "
       "because one model rewrote the entire corpus.")
if _stg.get("underpowered"):
    _A("- The confirmed coref-critical stratum is thin, so the stratified arm is a diagnostic "
       "only. The aggregate arm is the paper's own condition and does not depend on it.")
_A("")
if _incomplete:
    _A("## How to resume")
    _A("")
    _A("Nothing is lost. Every finished stage is checkpointed in `test-8-data/`.")
    _A("")
    _A("1. Save this kernel's **output**.")
    _A("2. Add it to `kernel-metadata.json` as a `kernel_sources` entry.")
    _A("3. Push again. `load_stage()` finds the checkpoints under `/kaggle/input` and resumes.")
    _A("")
    _A("No code changes are needed between pushes.")
else:
    _A("## Artifacts")
    _A("")
    _A("`test-8-results.md` carries every table and plot. `per_query_results.jsonl` is the "
       "durable raw artifact - it holds each query's top-50 doc ids, so any other metric can be "
       "recomputed on a CPU without re-running a GPU stage.")
_A("")

with open(os.path.join(_out, "RESULTS.md"), "w", encoding="utf-8") as f:
    f.write("\n".join(_lines))

print("\n".join(_lines))
print("=" * 88)
print("wrote", os.path.join(_out, "RESULTS.md"), "and run_summary.json")
print("ARTIFACTS:", sorted(os.listdir(_out)))


## Done

`/kaggle/working/test-8-data/` now holds every stage output, the raw
`per_query_results.jsonl`, the `delta_vs_baseline__*.png` plots, `test-8-results.md`, and the
`RESULTS.md` / `run_summary.json` pair written by the summary cell above — **start there**, it says
what ran, what every gate decided, what was skipped or quarantined, and how to resume if the
session budget cut the run short. **Save a notebook version so the outputs persist.**

The raw file is the durable artifact: it carries each query's top-50 doc ids, so any other metric
or cut can be recomputed later on a CPU without re-running a single GPU stage.

How to read the result — fixed in advance, before any numbers existed:

* **Effect only in the stratified subset** → dilution explains both this repo's null and the paper's
  small numbers.
* **Mean pooling gains, CLS does not** → their mechanism holds on BEIR-style data.
* **Both flat, stratified included** → coref-before-embed does not help dense retrieval here, and
  their small positives are within noise. A legitimate result; it will not be softened.
* **CIs overlap everywhere** → underpowered; state the N the power note gives.
* **A gain in the confirmed stratum that disappears in the `gold_has_pronoun` subset** → whatever
  moved, it was not coreference resolution.
* **Slope near −1 in the delta-vs-baseline regression** → regression to the mean, not a uniform
  improvement: coref only rescued queries the baseline was already failing.

And the standing caveat: if `corpus_pronoun_share` is low, a null is a property of scientific prose
rather than a finding about coreference. That number is in the results header for exactly this
reason.